# Extract features with the Claude Agent SDK (Opus 5.5) + tokens per call

One notebook for the whole pilot:
1. **Setup** — model, effort, paths.
2. **Sample** — builds the 100 stratified snapshots into `data/` (skipped if the file already exists).
3. **Prompt** — the full prompt as one string with placeholders, then the JSON schema built from its allowed values.
4. **Token estimate before running** — input tokens per snapshot and an estimated cost.
5. **Test on one snapshot** — check the answer and its tokens before spending on all 100.
6. **Run all snapshots, 10 in parallel** — one JSON per snapshot in `output/`; finished snapshots are skipped, so it is
   safe to stop and re-run.
7. **Tokens per call** — real input, output and thinking tokens and cost for every snapshot, from the saved JSONs.
8. **Compare with a previous run** — share of identical values per feature between `output/` and an earlier run's folder.

How a call works:
- The Agent SDK runs the Claude Code installed on this machine (`~/.local/bin/claude`) with its existing login, so no
  API key is needed.
- The fixed instructions go in as the system prompt (cached after the first call); the claim data goes in as the
  message.
- The answer comes back as structured output, then every value is checked again exactly (case-sensitive). A
  snapshot with a wrong value is retried once with the problems listed.

`data/` and `output/` contain raw notes and evidence quotes (PII) and are git-ignored.

## 1. Setup

In [1]:
%pip install -q claude-agent-sdk==0.2.180   # SageMaker wipes /opt/conda packages on restart, so install every time
import glob
import json
import os
import re
import tempfile
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query

pd.set_option('display.max_columns', None)

MODEL = 'claude-opus-5-5'
EFFORT = 'high'                  # low / medium / high / xhigh / max
MAX_TURNS = 4                    # the answer is returned through the SDK's StructuredOutput tool, so allow a few turns
MAX_BUDGET_USD_PER_CALL = 3.0    # safety stop for any single snapshot
MAX_ATTEMPTS = 2                 # 1 retry if a value is not an exact allowed value
import sys; sys.path.insert(0, '..')   # repo root, so config.py can be imported
from config import CLAUDE_CODE_PATH as CLAUDE_CODE   # the installed, logged-in Claude Code

SAMPLE_PATH = 'data/pilot_sample_100.parquet'
OUTPUT_DIR = 'output'
SYSTEM_PROMPT_FILE = os.path.abspath('prompt_cache/system_prompt.md')   # absolute: Claude Code runs in an empty temp folder

# Claude Opus 5.5 list prices, USD per million tokens. Claude Code caches with the 1-hour TTL (2x input).
PRICES = {'input': 4.00, 'output': 20.00, 'cache_write_5m': 5.00, 'cache_write_1h': 8.00, 'cache_read': 0.20}

Note: you may need to restart the kernel to use updated packages.


## 2. Build the 100-snapshot sample
Build the 100-snapshot pilot sample for LLM feature extraction (Claude Agent SDK, Opus 5.5).

Per-model balanced sample, all from the TRAIN split (holdout stays untouched):
  - Attorney   34 snapshots: 17 with an attorney, 17 without
  - Litigation 33 snapshots: 17 litigated, 16 not
  - Severity   33 snapshots: severity buckets 0-4 as equal as possible (7/7/7/6/6)
Within each target class, rows are spread across ORG_GROUP_NAME x POL_STATE x LOSS_YEAR
(the same columns 3.1-Dataset-Development stratifies on) by systematic sampling over a
stratum-sorted, shuffled list. One coverage per claim, and no claim is reused across models.
Each chosen coverage contributes its last snapshot in that model's dataset (the most notes before the cutoff).

Writes `data/pilot_sample_100.parquet` (raw notes, git-ignored) and `data/pilot_sample_summary.csv` (counts only).

In [2]:
import pyarrow.dataset as ds
import pyarrow.fs as pafs

import sys; sys.path.insert(0, '..')   # repo root, so config.py can be imported
from config import TARGET_DIR, S3_BUCKET, SNAPSHOT_PREFIX, AWS_REGION   # paths: config.py


def build_sample():
    SEED = 42
    OUT_DIR = 'data'
    S3_ROOT = f'{S3_BUCKET}/{SNAPSHOT_PREFIX}'   # bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets
    PLAN = {  # model -> {target value: number of snapshots}
        'Attorney':   {1: 17, 0: 17},
        'Litigation': {1: 17, 0: 16},
        'Severity':   {0: 7, 1: 7, 2: 7, 3: 6, 4: 6},
    }

    rng = np.random.default_rng(SEED)
    fs = pafs.S3FileSystem(region=AWS_REGION)


    def train_dataset(model):
        return ds.dataset(f'{S3_ROOT}/{model}/train', format='parquet', filesystem=fs)


    # coverage attributes (stratification columns + prompt metadata)
    cov = pd.read_parquet(os.path.join(TARGET_DIR, 'BI-Coverage-ALL-2015_2022.parquet'))
    sev = pd.read_parquet(os.path.join(TARGET_DIR, 'Endpoint-Severity-2015_2022.parquet'))
    cov = cov.merge(sev[['CLM_CVG_PK', 'SEV_BUCKET']], on='CLM_CVG_PK', how='left')
    ATTR = ['CLM_CVG_PK', 'ORG_GROUP_NAME', 'POL_STATE', 'LOSS_YEAR', 'LOSS_STATE', 'BI_TYPE',
            'LOSS_DT', 'CLM_OPEN_DT', 'CVG_OPEN_DT', 'ATTNY_FLG', 'LITIG_FLG', 'SEV_BUCKET']

    used_claims = set()
    picked = []
    for model, plan in PLAN.items():
        # snapshot index of the train split (small columns only; NOTE_TEXT is not read here)
        target_col = 'TARGET_BUCKET' if model == 'Severity' else 'TARGET'
        idx = train_dataset(model).to_table(
            columns=['CLM_CVG_PK', 'CLM_NUM', 'SNAPSHOT_NUM', 'SNAPSHOT_DAY', 'N_NOTES', target_col]).to_pandas()
        idx = idx.rename(columns={target_col: 'TARGET'})
        cvgs = idx.groupby('CLM_CVG_PK').agg(CLM_NUM=('CLM_NUM', 'first'), TARGET=('TARGET', 'first')).reset_index()
        cvgs = cvgs.merge(cov[ATTR], on='CLM_CVG_PK', how='left')
        cvgs = cvgs[~cvgs.CLM_NUM.isin(used_claims)]
        cvgs = cvgs.sample(frac=1, random_state=SEED).drop_duplicates('CLM_NUM')  # one coverage per claim
        cvgs['STRATUM'] = (cvgs.ORG_GROUP_NAME.astype(str) + '|' + cvgs.POL_STATE.astype(str) + '|'
                           + cvgs.LOSS_YEAR.astype(str))
        for target_value, n in plan.items():
            pool = cvgs[cvgs.TARGET == target_value]
            pool = pool.assign(_r=rng.random(len(pool))).sort_values(['STRATUM', '_r'])
            step = len(pool) / n
            start = rng.random() * step
            chosen = pool.iloc[[int(start + i * step) for i in range(n)]]
            for _, row in chosen.iterrows():
                snaps = idx[idx.CLM_CVG_PK == row.CLM_CVG_PK]
                snap = snaps.loc[snaps.SNAPSHOT_NUM.idxmax()]   # the coverage's last snapshot
                picked.append({'MODEL': model, 'CLM_CVG_PK': int(row.CLM_CVG_PK), 'CLM_NUM': row.CLM_NUM,
                               'SNAPSHOT_NUM': int(snap.SNAPSHOT_NUM), 'SNAPSHOT_DAY': int(snap.SNAPSHOT_DAY),
                               'N_SNAPSHOTS_AVAILABLE': len(snaps), 'TARGET': int(row.TARGET)})
                used_claims.add(row.CLM_NUM)
        print(f'{model}: picked {sum(plan.values())} from {len(cvgs):,} eligible train coverages')

    sample = pd.DataFrame(picked)

    # pull NOTE_TEXT for exactly the chosen (coverage, snapshot) rows
    texts = []
    for model, grp in sample.groupby('MODEL'):
        t = train_dataset(model).to_table(
            columns=['CLM_CVG_PK', 'SNAPSHOT_NUM', 'N_NOTES', 'NOTE_TEXT'],
            filter=ds.field('CLM_CVG_PK').isin(grp.CLM_CVG_PK.tolist())).to_pandas()
        t = t.merge(grp[['CLM_CVG_PK', 'SNAPSHOT_NUM']], on=['CLM_CVG_PK', 'SNAPSHOT_NUM'])
        texts.append(t.assign(MODEL=model))
    sample = sample.merge(pd.concat(texts), on=['MODEL', 'CLM_CVG_PK', 'SNAPSHOT_NUM'], how='left')
    assert sample.NOTE_TEXT.notna().all() and len(sample) == 100

    # prompt metadata (no target information) + the two computed features
    sample = sample.merge(cov[ATTR], on='CLM_CVG_PK', how='left')
    sample['SNAPSHOT_DATE'] = (pd.to_datetime(sample.CLM_OPEN_DT) + pd.to_timedelta(sample.SNAPSHOT_DAY, unit='D')).dt.date
    sample['LOSS_DATE'] = pd.to_datetime(sample.LOSS_DT).dt.date
    sample['CLAIMANT'] = 'not provided'
    days = (pd.to_datetime(sample.CVG_OPEN_DT) - pd.to_datetime(sample.LOSS_DT)).dt.days
    sample['A60'] = sample.BI_TYPE.where(sample.BI_TYPE.isin(['BI', 'UMBI', 'UIMBI']))
    sample['L70'] = np.select([days.isna(), days <= 6, days <= 30, days <= 90],
                              ['Date missing', 'Same week (0-6 days)', '1-4 weeks', '1-3 months'], '3+ months')
    sample['NOTE_CHARS'] = sample.NOTE_TEXT.str.len()

    cols = ['MODEL', 'TARGET', 'CLM_CVG_PK', 'CLM_NUM', 'SNAPSHOT_NUM', 'SNAPSHOT_DAY', 'N_SNAPSHOTS_AVAILABLE',
            'N_NOTES', 'NOTE_CHARS',
            # prompt placeholders
            'SNAPSHOT_DATE', 'LOSS_DATE', 'LOSS_STATE', 'POL_STATE', 'BI_TYPE', 'CLAIMANT', 'NOTE_TEXT',
            # computed features
            'A60', 'L70',
            # stratification + reference labels (NEVER put these in the prompt)
            'ORG_GROUP_NAME', 'LOSS_YEAR', 'ATTNY_FLG', 'LITIG_FLG', 'SEV_BUCKET']
    sample = sample[cols].sort_values(['MODEL', 'TARGET', 'CLM_CVG_PK']).reset_index(drop=True)
    os.makedirs(OUT_DIR, exist_ok=True)
    sample.to_parquet(os.path.join(OUT_DIR, 'pilot_sample_100.parquet'), index=False)

    summary = pd.concat([
        sample.groupby(['MODEL', 'TARGET']).size().rename('n').reset_index().assign(column='TARGET'),
        *[sample.groupby(['MODEL', c]).size().rename('n').reset_index().rename(columns={c: 'TARGET'}).assign(column=c)
          for c in ['ORG_GROUP_NAME', 'POL_STATE', 'LOSS_YEAR']],
    ]).rename(columns={'TARGET': 'value'})[['MODEL', 'column', 'value', 'n']]
    summary.to_csv(os.path.join(OUT_DIR, 'pilot_sample_summary.csv'), index=False)
    print(sample.groupby(['MODEL', 'TARGET']).size())
    print(f'notes per snapshot: median {sample.N_NOTES.median():.0f}, max {sample.N_NOTES.max()}; '
          f'chars median {sample.NOTE_CHARS.median():,.0f}, max {sample.NOTE_CHARS.max():,}')


if os.path.exists(SAMPLE_PATH):
    print(f'{SAMPLE_PATH} already exists - using it (delete it to rebuild).')
else:
    build_sample()


data/pilot_sample_100.parquet already exists - using it (delete it to rebuild).


## 3. Prompt, schema and helpers

In [3]:
PROMPT = r'''# Task

You are a senior auto bodily-injury (BI) claims analyst at a US auto insurer. You will read the adjuster notes for one BI claim as they stood on a snapshot date, and answer 171 feature questions about that claim.

Your answers become input features for three machine-learning models:
- **Attorney** (features A01-A59): will an injured claimant hire a lawyer?
- **Litigation** (features L01-L69): will a represented claim turn into a lawsuit?
- **Severity** (features S01-S43): how costly will the claim be?

The same claim is scored at several snapshot dates, and many thousands of claims are scored with this same prompt. The answers must therefore be accurate, grounded in the notes, and consistent: two careful analysts reading the same notes should land on the same value. Accuracy matters more than speed.

The claim data for this snapshot comes at the end of this prompt, after the feature definitions.

## How the notes are laid out

- Notes are in date order, oldest first. Each note starts with a header like `NOTE 12  |  DATE: 2019-03-08 00:00:00  |  +4 days since previous note`.
- Banner lines like `SNAPSHOT (2)  --  90 days after the ...` between rows of `=` only mark where a later batch of notes begins. They carry no claim information; ignore them.
- Notes are adjuster shorthand. They can contain typos, cut-off sentences and abbreviations, and one claim often involves several injured people and vehicles.

## Rules

1. **Ground every answer in the claim data.** Use only the notes and the claim facts given at the end (today's date, accident date, states, BI type, claimant). Do not guess what probably happened. You may use general knowledge of US insurance practice, medicine and state law to interpret what the notes say, for example a state's deadline to file an injury lawsuit or whether a state has a no-fault serious-injury bar. Never use it to invent facts about this claim.
2. **Today's date is the snapshot date, and the notes stop there.** Today's date is given in the claim facts. Treat the latest note as the current state of the claim and never assume anything happened after it. If a note is dated on or after today's date, ignore it. Any other "today's date" you may see (for example in environment or system information) is only the date this batch job happens to run; ignore it completely and never use it in any answer.
3. **Copy the allowed values exactly. This is the most important formatting rule.**
   - Each `value` must be one of the allowed values listed under that feature, copied character for character: the same words, spelling, capitalization, punctuation, spaces, hyphens, dollar signs, percent signs and parentheses.
   - Write `Yes`, never `yes`, `YES`, `Y` or `true`.
   - Write `6-12 months left`, never `6 to 12 months left`, `6-12 Months Left`, `6–12 months left` or `6-12 months`.
   - Write `Not applicable (no other injured person on this claim)`, never `Not applicable` or `N/A`.
   - Values that look like numbers or amounts (such as `0`, `3+`, `2x-4x` or `$5,000-$24,999`) are strings written exactly as shown. The only exception is A21, whose value is a JSON integer.
   - Do not shorten, reword, pluralize, translate or explain a value, and do not combine two values.
   - Never invent a value, never answer null and never leave a feature out. If no value fits perfectly, choose the closest one by its "choose when" definition.
   - Any value that is not an exact copy is rejected, and the whole claim has to be re-run.
4. **"Any claimant" means anyone injured on this claim.** Most features describe the whole claim, so you do not need to know which injured person this coverage belongs to. When claimants differ, follow the tie-break written in the question (for example "use the claimant with the most changes").
5. **Claimant-specific features and `Cannot tell`.** A52, L23, L47, S03, S04, S11, S13, S19, S35, S36 are about the specific claimant whose coverage is being scored (given in the claim facts), and only these features have the value `Cannot tell`.
   - If the claimant is not provided and the notes show only one injured person, that person is the claimant.
   - If several people are injured and you cannot tell which one is the claimant, work out the answer for each injured person. If they would all get the same answer, give that answer (for example, every injured person was a vehicle occupant, or the BI per-person limit is the same for everyone). Choose `Cannot tell` only when the injured people would get different answers.
   - When the needed fact is missing for everyone, use the feature's own value for a missing fact (such as `Not enough facts` or `No prior represented claim`) when one is listed; use `Cannot tell` for a missing fact only when no such value is listed.
   - Never use `Cannot tell` for any other feature.
6. **Tie-breaks.**
   - "If several fit, pick the highest" means choose the most serious or most advanced level, as described in the question and the value definitions.
   - "Pick the one listed last" and "the first that applies" refer to the order of the allowed values as listed under that feature. When the question itself gives an order, use the question's order.
7. **When the notes are silent.**
   - If the question says what to answer when something isn't mentioned, do that.
   - Otherwise, when nothing in the notes points to another value, choose the value whose definition describes absence (such as `No`, `None`, `None noted` or `Not represented`).
   - Choose `Not enough facts`, `Not enough information`, `Not enough contacts` or `Too early to tell` only when that value is listed and the notes lack what its definition requires.
   - Treat a fact as known when any note states it in any form: a police-report summary, a first-report form, a later note describing an earlier event, or common shorthand. Do not choose a `Not enough ...` value just because the notes are brief; choose it only in the narrow case its definition (or the question) gives.
8. **Dates, windows and counts.**
   - Use the dates on the note headers, and count calendar days.
   - "Latest N days" means the N days ending at the last note.
   - "First N days", "within N days" and "by day N" count from the accident date (day 0). Use only notes dated inside that window, even if later notes exist.
   - If the snapshot falls before the window ends, use the notes available.
   - For deadlines, use the accident date and the accident state.
9. **Who is who.**
   - "Our company" or "we" is the insurer whose adjusters wrote these notes, and "our insured" is its policyholder.
   - On a BI claim, the claimants are people hurt by our insured's vehicle (IV).
   - On a UMBI or UIMBI claim, the claimant is claiming on their own policy with us, and the at-fault driver is in the other vehicle (OV).
10. **Common abbreviations:** IV = our insured's vehicle; OV = the other vehicle; IVD/OVD = IV/OV driver; IVP/OVP, PIV/POV = passenger in the IV/OV; DIV/DOV = driver of the IV/OV; clmt = claimant; insd/NI = our (named) insured; BI = bodily-injury liability; UM/UIM = uninsured/underinsured motorist; UMBI/UIMBI = a claim under the injured person's own UM/UIM coverage; PIP/NF = no-fault personal injury protection; med pay = medical payments coverage; LOR = letter of representation from a lawyer; atty/PA = attorney; PL = policy limits; SOL = statute of limitations (deadline to sue); ROR = reservation of rights; DOL/DOA = date of loss/accident; FNOL = first notice of loss; PR = police report; SIU = special investigations unit; ISO = prior-claims search; ER = emergency room; PT = physical therapy; chiro = chiropractor; IME = independent medical exam; NCM = nurse case manager; UR = utilization review; MRI/CT/EMG = imaging and nerve tests; LOC = loss of consciousness; TBI = traumatic brain injury; DOB = date of birth; lm/vm = left message/voicemail; cb = call back; TL = total loss; liab = liability; eval = evaluation; auth = settlement authority or medical authorization (from context).
11. **Keep related answers consistent.** Several features look at the same facts from different angles, so their answers must agree. For example, if no claimant has a lawyer, every feature that lists `Not represented` must use it. And if no offer has been made, every offer feature must use its "no offer" value.
12. **Evidence.** For each feature, first write `evidence`: a short quote or close paraphrase from the notes, at most 25 words, starting with its note date (for example `2019-03-08: spoke w/ clmt, treating chiro 3x/wk`). Then choose `value`. Use an empty string for evidence when the value means something is absent.
13. **Output only the JSON object** described in the output format, with no other text.

# Feature definitions

## Part 1: Attorney features (A01-A59)

### A01: Another injured person on this same claim already has a lawyer

**What it asks:** Whether some OTHER injured person on this same claim (not necessarily the claimant being scored) already has a lawyer. Look for letters of representation (LOR), 'atty', 'repped', or attorney contact tied to another injured person.

**Question:** As of the snapshot date, has any other injured person covered under this same accident claim already hired an attorney?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when another injured person on the claim has a lawyer (letter of representation or lawyer contact noted)
- `No`: choose when other injured people exist, none has a lawyer yet
- `Not applicable (no other injured person on this claim)`: choose when this claimant is the only injured person

### A02: Another injured person on this same claim is already suing

**What it asks:** Whether some OTHER injured person on this claim has already filed a lawsuit. Look for suit papers, summons or complaint received for another injured person.

**Question:** As of the snapshot date, has any other injured person covered under this same accident claim already filed a lawsuit?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when another injured person on the claim has filed suit
- `No`: choose when other injured people exist, none has sued yet
- `Not applicable (no other injured person on this claim)`: choose when this claimant is the only injured person

### A03: Has our adjuster ever talked directly with any claimant

**What it asks:** Whether our adjuster ever had a real two-way conversation with any injured claimant (not only voicemails, letters, or talking through their lawyer or a relative). Look for 'spoke with clmt', 'spk w/', a completed call.

**Question:** Has our adjuster ever had a live two-way conversation with any injured claimant on this claim, rather than only through a lawyer or a relative? A conversation with the claimant through an interpreter counts as direct.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when at least one live call or meeting with a claimant, including through an interpreter
- `No`: choose when only voicemails, letters, or contact through a lawyer or relative

### A04: Are the claimants easy to reach

**What it asks:** How responsive the claimants are to our outreach overall. Look at call logs: 'lm', 'vm full', 'no answer', 'cb rec'd', and the dates between our attempt and their reply.

**Question:** Are the injured claimants on this claim easy to reach? Easy means most of our calls, texts or emails to them get an answer or call back within about two business days, and never 3 or more unanswered attempts in a row. Count each call, voicemail, text, email or contact letter to a claimant as one attempt. Decide in this order: `No` if any claimant had 3+ unanswered attempts in a row, or most replies took longer than two business days; otherwise `Not enough contacts` if fewer than 3 attempts in total were made to claimants; otherwise `Yes`.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when every claimant we tried usually answers within about two business days, never 3+ unanswered in a row
- `No`: choose when at least one claimant had 3+ unanswered attempts in a row, or usually took longer than two business days
- `Not enough contacts`: choose when fewer than 3 contact attempts to any claimant noted

### A05: Did we lose contact with any claimant after an adverse action

**What it asks:** Whether a claimant went silent after we did something unfavorable to them (denial, low offer, cut-off of treatment or benefits). First find adverse actions, then check the claimant's responses afterward.

**Question:** After an adverse action by our company, such as a denial, a low offer or a cut-off of treatment or benefits, did that claimant stop responding? Stopping means no reply to 3+ attempts or for 14+ days after the action.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when after an adverse action, the affected claimant did not reply to 3+ attempts or for 14+ days
- `No`: choose when claimants kept responding after every adverse action
- `Adverse action not happened`: choose when no denial, low offer or cut-off is noted

### A06: Did any claimant stay in touch but turn rude after an adverse action

**What it asks:** Whether a claimant kept talking to us after an adverse action but became rude, angry or hostile. Look for yelling, insults, threats or hostile tone noted after a denial, low offer or cut-off.

**Question:** After an adverse action by our company, did any claimant keep talking to us but become rude, angry or hostile, for example yelling, insulting or threatening?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when after an adverse action, a claimant kept in contact but was rude, angry or hostile
- `No`: choose when claimants stayed civil after adverse actions
- `Adverse action not happened`: choose when no denial, low offer or cut-off is noted

### A07: Did our adjuster go more than 10 days without following up

**What it asks:** Whether our adjuster let something pending sit for more than 10 days without any follow-up. Look for pending callbacks, documents or decisions and the dates of our next action.

**Question:** Was there a gap of more than 10 days in which our adjuster made no follow-up while something was pending, such as a callback, a document or a decision? Count calendar days between dated notes.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a gap of 11+ days with no adjuster follow-up while something was pending
- `No`: choose when no such gap, or gaps happened only when nothing was pending

### A08: Is our company proactive or reactive

**What it asks:** Whether our adjuster drives the claim forward on their own or mainly reacts to incoming contacts. Count our outgoing actions and whether each followed an incoming contact within 2 business days.

**Question:** Count our outgoing actions (calls, letters, offers, requests) and whether each was prompted by an incoming contact in the 2 business days before. Is our adjuster mostly moving the claim forward unprompted? An action is prompted when an incoming contact (call, letter, email, document or bill) from anyone outside our company came in the 2 business days before; actions within 2 business days after the first report also count as prompted. Count calls, letters, requests and offers only; diaries and internal reviews don't count.

**Allowed values (copy exactly one, character for character):**
- `Proactive`: choose when more than half of our actions were not prompted by an incoming contact
- `Reactive`: choose when half or fewer of our actions were unprompted

### A09: Is any claimant's injury getting worse or better over time

**What it asks:** The direction of the injuries over time: escalating (new specialists, damaging scans, injections, surgery), steady, or improving. Compare the earliest and latest injury and treatment descriptions.

**Question:** Comparing the earliest description of each claimant's injury and treatment with the most recent, is it getting worse, staying the same or getting better? If claimants differ, pick Getting worse if anyone is, otherwise Staying the same if anyone is.

**Allowed values (copy exactly one, character for character):**
- `Getting worse`: choose when new specialist, a scan showing a fracture, herniation or tear, injections or surgery recommended
- `Staying the same`: choose when same complaints and same level of care
- `Getting better`: choose when improving, care reduced, or discharged
- `Not enough facts`: choose when only one description of the injury, or none

### A10: How the claimants act toward the insurance company

**What it asks:** The claimants' attitude toward our company, mainly in the latest 60 days of direct contact: friendly, neutral, or complaining and hostile.

**Question:** Judging mainly from the latest 60 days of direct contact, what is the claimants' attitude toward our company? If claimants differ, pick Negative if anyone is negative, otherwise Neutral if anyone is neutral. Attitude comes only from two-way contact with a claimant or a family member speaking for them (our insured counts only on a UMBI or UIMBI claim). `Positive` needs an explicit sign such as thanks, praise or friendly words noted; simply agreeing or answering questions is `Neutral`.

**Allowed values (copy exactly one, character for character):**
- `Positive`: choose when thanks us, cooperative or friendly in most contacts
- `Negative`: choose when complains, is angry or accusing, or refuses requests in most contacts or the latest contact
- `Neutral`: choose when neither positive nor negative, just answers questions
- `Not enough facts`: choose when no direct contact with any claimant

### A11: Did we give any claimant something extra before they had to ask

**What it asks:** Whether we gave any claimant something not required or before they asked (extra rental days, waived fee, faster payment, covering an out-of-pocket cost).

**Question:** Did the adjuster offer or give any claimant something not required by the policy, or before it was asked for, such as extra rental days, waiving a fee, paying faster than required, or covering an out-of-pocket cost?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when at least one such extra was offered or given without the claimant asking first
- `No`: choose when we gave only what was required or requested

### A12: Was a manager brought in early because the claim looked serious, or only after something went wrong

**What it asks:** Whether a manager, special unit or senior reviewer got involved, and if so why: early because the case is truly severe (within 30 days, with severity, limits or exposure given as the reason), only after a problem, or just for routine reporting. Early involvement for a general report or routine talk is routine, not severity.

**Question:** If a manager, special unit or senior reviewer got involved, why? Count it as early due to severity only when they joined within 30 days of first notice and the notes tie their involvement to the real seriousness of the case (injury severity, policy limits or a large exposure). A manager joining early only for a general report, status update, new-claim review or routine discussion does not count as severity. Otherwise, was it only after a complaint, dispute or problem had already happened?

**Allowed values (copy exactly one, character for character):**
- `Brought in early due to severity`: choose when involved within 30 days of first notice, before any complaint or dispute, and the notes say it was because of injury severity, policy limits or a large exposure (a general report or routine talk does not count)
- `Only brought in after a problem`: choose when involved only after a complaint, dispute or mistake
- `Routine sign-off only`: choose when only a scheduled review, routine approval, general report, status update or routine discussion with no severity reason given, even if it happened early
- `Not involved yet`: choose when no manager, special unit or senior reviewer noted

### A13: Does any claimant's injury have objective medical proof, or just the claimant's word

**What it asks:** Whether any claimant's injury is backed by objective medical evidence (fracture, MRI or CT showing herniation or tear, documented loss of consciousness, recommended surgery), or only by complaints.

**Question:** Does any claimant's injury have objective medical proof, such as a fracture, an MRI or CT showing a herniation or tear, documented loss of consciousness, or a recommended surgery? If claimants differ, pick Real medical proof exists.

**Allowed values (copy exactly one, character for character):**
- `Real medical proof exists`: choose when at least one claimant has one of the listed objective findings
- `Only claimant's word, no proof`: choose when complaints only, with normal or no test results
- `Not enough facts`: choose when no injury or test details in the notes

### A14: Was the IV driver drunk

**What it asks:** Whether our insured's driver (IV driver) was drunk or impaired. Look at the police report summary, charges, tests, or the driver's own admission.

**Question:** Was the IV driver, the driver of our insured's vehicle, drunk or impaired by alcohol or drugs at the time of the accident? If the notes don't mention impairment, answer No.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a police report, charge, test or admission says the IV driver was drunk, drugged or impaired
- `No`: choose when not mentioned, or ruled out

### A15: Was the OV driver drunk

**What it asks:** Whether the other vehicle's driver (OV driver) was drunk or impaired. Look at the police report summary, charges, tests, or admissions.

**Question:** Was the OV driver, the driver of the other vehicle, drunk or impaired by alcohol or drugs at the time of the accident? If the notes don't mention impairment, answer No.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a police report, charge, test or admission says the OV driver was drunk, drugged or impaired
- `No`: choose when not mentioned, or ruled out

### A16: How serious the injuries in this accident looked

**What it asks:** How serious the worst injury to anyone in the accident looked in the first 7 days after the accident, judged only on hard facts (ambulance, ER or hospital, diagnosis, admission), not on described pain.

**Question:** Based on hard facts about the first 7 days after the accident (ambulance, ER or hospital visit, diagnosis, admission), how serious was the worst injury to anyone in this accident, regardless of which claimant? Count what happened in those 7 days even when a later note reports it (for example, a note on day 30 saying the claimant went to the ER on the accident day). "Taken to", "transported to" or "went to" a hospital or medical center counts as an ER visit. Ignore described pain levels. If more than one level fits, pick the highest. When someone says they are hurt but no note mentions an ambulance, ER or hospital visit, or a diagnosis in those 7 days, answer `Mild`.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when the notes say no one was hurt, or everyone declined medical care
- `Mild`: choose when someone reports pain, soreness, stiffness, bruises or minor scrapes, and no note mentions an ambulance, ER or hospital visit, or diagnosis in the first 7 days (no care, urgent care or own doctor only)
- `Moderate`: choose when anyone went by ambulance or to an ER or hospital in the first 7 days and was sent home the same day, or has a sprain, strain, whiplash, cut needing stitches, or concussion without loss of consciousness
- `Severe`: choose when anyone has a fracture, dislocation, internal injury, head injury with loss of consciousness, was admitted overnight, had surgery, or died
- `Not enough facts`: choose when no note says whether anyone was hurt in this accident yet

### A17: How severe the crash was

**What it asks:** How severe the crash was, judged on the most damaged vehicle and hard facts (damage, towing, airbags, injuries at the scene).

**Question:** Based on the most damaged vehicle and hard facts (damage, towing, airbags, injuries at the scene), not on what anyone says about pain, how severe was the crash? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `Mild`: choose when drivable, scratches or dents, no airbags
- `Moderate`: choose when towed or not drivable, or airbags deployed, but repairable
- `Severe`: choose when total loss, rollover, someone ejected or trapped, or a pedestrian or cyclist struck
- `Not enough facts`: choose when notes don't describe vehicle damage or towing

### A18: Is contact with the claimants speeding up or slowing down recently

**What it asks:** Whether our contact with the claimants is getting more frequent or less frequent: count contacts in the latest 60 days versus the 60 days before.

**Question:** Across all claimants on this claim, compare the number of contacts (calls, texts, emails, letters, visits) in the latest 60 days with the 60 days before that. Use the real dates of notes. Count each call, voicemail, text, email, letter or visit to or from a claimant as one contact, answered or not; contacts only with a lawyer or our insured don't count. If there were no contacts in the earlier 60 days and 3+ in the latest 60 days, answer `Speeding up`.

**Allowed values (copy exactly one, character for character):**
- `Speeding up`: choose when 25% or more contacts in the latest 60 days
- `About the same`: choose when within 25% either way
- `Slowing down`: choose when 25% or fewer contacts in the latest 60 days
- `Not enough contacts to tell`: choose when fewer than 3 contacts in the two windows combined

### A19: Was anyone in the IV denied a treatment

**What it asks:** Whether any insurer denied, cut off or refused to pay for a test, treatment or bill for someone injured in our insured's vehicle (IV).

**Question:** Did an insurer deny, cut off or refuse to pay for a medical test, treatment or bill for anyone injured in the IV (our insured's vehicle)? A request for more paperwork is not a denial.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when an insurer denied, cut off or refused to pay a test, treatment or bill for someone in the IV
- `No`: choose when people in the IV were hurt, no denial noted
- `No one injured in the IV`: choose when nobody in the IV claimed injury

### A20: Did the person in the IV fight the treatment denial

**What it asks:** If someone in the IV was denied treatment, whether they, their doctor or family appealed, disputed or took it to arbitration.

**Question:** If someone injured in the IV was denied a treatment, did they, their doctor or their family appeal it, dispute it or take it to arbitration?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when the person, their doctor or family appealed, disputed or took the denial to arbitration
- `No`: choose when the denial was accepted with no appeal or dispute
- `No denial in the IV`: choose when no one in the IV was denied a treatment

### A21: How many days fault stayed disputed between the cars

**What it asks:** How long fault between the IV and OV stayed disputed, in days, from the first note showing a dispute to the note where fault was agreed.

**Question:** If fault between the IV and the OV was disputed, how many days passed from the first note showing the dispute to the note where fault was agreed, by the parties or by an arbitrator? Use -1 if fault was never disputed and -99 if it is still disputed.

**Allowed values (this is the only feature whose value is a JSON integer, not a string):**
- `-1`: choose when fault was never disputed
- `-99`: choose when fault is still disputed at the latest note
- a whole number of days, 0 or more (for example `42`): choose when fault was disputed and later agreed; it is the days from the first dispute note to the agreement note. Write the number itself, never the words "Number of days".

### A22: Has any claimant filed similar injury claims before, to the same body part

**What it asks:** The worst prior-claims history of any claimant from ISO or similar searches quoted in the notes, and whether any prior claim involved a body part injured now.

**Question:** From the prior-claims searches in the notes (ISO or similar), what is the highest prior-claims level for any claimant on this claim, counting earlier injury claims and whether any involves a body part injured now? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None found`: choose when searches ran and show no earlier injury claims for any claimant
- `Prior claims, different body parts only`: choose when earlier injury claims exist, none to a body part injured now
- `At least one prior claim to the same body part`: choose when one or two earlier claims include a body part injured now
- `Three or more, including the same body part`: choose when one claimant has 3+ earlier injury claims, one to a body part injured now
- `Not enough facts`: choose when no prior-claims search result is mentioned in the notes

### A23: Did any claimant have a lawyer or a lawsuit on a past claim

**What it asks:** Whether any claimant had a lawyer, or a lawsuit, on an earlier, separate injury claim, based on prior-claims searches in the notes.

**Question:** From the prior-claims searches in the notes (ISO or similar), did any claimant on this claim have a lawyer, or a lawsuit, on an earlier, separate injury claim? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No prior claims on file`: choose when searches show no earlier injury claims for any claimant
- `Prior claims, no lawyer`: choose when earlier claims exist, none shows a lawyer or lawsuit
- `Had a lawyer on a prior claim`: choose when an earlier claim shows a lawyer, but no lawsuit
- `A prior claim went to a lawsuit`: choose when an earlier claim shows a lawsuit was filed
- `Not enough facts`: choose when no prior-claims search result is mentioned in the notes

### A24: Did our own staff admit a mistake that delayed any claimant

**What it asks:** Whether our notes admit a staff mistake (missed referral, form, payment or callback) that delayed a claimant's payment, benefit or decision by more than 14 days.

**Question:** Do our notes admit that something our staff should have done (a referral, a form, a payment or a callback) was not done, and that it delayed a payment, benefit or decision for any claimant on this claim by more than 14 days? Small slips fixed within a few days don't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a note admits our miss and a claimant waited more than 14 days because of it
- `No`: choose when no admitted staff miss, or every miss was fixed within 14 days

### A25: Did any claimant's own no-fault medical money run out while still treating

**What it asks:** Whether any claimant's own no-fault (PIP) or medical-payments coverage was used up while they were still treating (a visit within the 30 days before).

**Question:** Has any claimant's own no-fault (PIP) or medical-payments coverage been used up, or a notice sent saying so, while that claimant was still treating, meaning a treatment visit in the 30 days before?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when PIP or med-pay used up, or exhaustion notice sent, with a treatment visit in the prior 30 days
- `No`: choose when not used up, not treating at that time, or no such coverage

### A26: How many people got hurt in this one accident

**What it asks:** How many different people were reported injured in this accident, across all vehicles and pedestrians.

**Question:** How many different people were reported injured in this accident, counting drivers, passengers and pedestrians in all vehicles?

**Allowed values (copy exactly one, character for character):**
- `1`: choose when only one person reported injured
- `2`: choose when two different people reported injured
- `3`: choose when three different people reported injured
- `4+`: choose when four or more different people reported injured

### A27: Is this a complicated multi-car or multi-insurance-policy claim

**What it asks:** How complicated the claim is: number of vehicles involved and number of insurance policies paying the injured.

**Question:** How many vehicles were involved, and how many insurance policies have to pay the injured people? Count a policy only if the notes show it was asked to pay or confirmed coverage. If several fit, pick the highest. Count only policies that pay injury damages: BI liability, UM/UIM, and excess or umbrella. Do not count PIP, med-pay or health insurance. On a UIMBI claim the at-fault driver's BI policy and our UIM policy are two paying policies.

**Allowed values (copy exactly one, character for character):**
- `Simple`: choose when one or two vehicles in total and one paying policy
- `A third vehicle or a second policy`: choose when three vehicles in total, or two paying policies
- `Four or more vehicles, or three or more policies`: choose when four or more vehicles in total, or three or more paying policies

### A28: Does any claimant say this injury is a flare-up of an old injury

**What it asks:** Whether any claimant describes the injury as a flare-up or worsening of a condition they had before the accident.

**Question:** Does any claimant on this claim describe their current symptoms as a flare-up or worsening of a condition they had before the accident, such as an earlier injury, surgery or diagnosis to the same area, rather than a new injury?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant says an old injury, surgery or condition was made worse
- `No`: choose when no claimant links current symptoms to a pre-accident condition

### A29: Is a relative negotiating for any claimant, not just passing along messages

**What it asks:** Whether a relative or friend (not a lawyer) is negotiating, setting conditions or refusing our requests for a claimant, beyond relaying messages or translating.

**Question:** Is a relative or friend (not a lawyer) negotiating amounts, setting conditions or refusing our requests, such as a medical authorization or a statement, for any claimant on this claim? Only relaying messages, translating, or a parent handling a child's claim doesn't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a relative or friend negotiates money, sets conditions or refuses our requests for a claimant
- `No`: choose when relatives only relay messages, translate, act for a child, or aren't involved

### A30: Is our company treating any claimant with suspicion (fraud or low impact)

**What it asks:** Whether we opened a fraud or SIU review, or noted that the impact looks too minor for the injuries claimed.

**Question:** Have we opened a fraud or special-investigations review, or noted that the impact looks too minor for the injuries claimed on this claim? Coverage and eligibility questions are covered by other features. Judge this apart from how serious the injuries are.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a fraud or SIU referral, or a low-impact concern, is noted
- `No`: choose when neither is noted

### A31: Did a repair shop or doctor's office have to repeatedly chase our company

**What it asks:** Whether a body shop, rental company or medical provider had to contact us two or more times about the same unresolved issue.

**Question:** Did a body shop, rental company or medical provider contact us two or more times about the same unresolved issue, such as an unpaid bill, an estimate or an authorization, before we acted?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when the same shop, rental company or provider contacted us 2+ times on one open issue
- `No`: choose when no vendor or provider had to contact us more than once per issue

### A32: Did any claimant agree to a settlement amount and then go silent on the paperwork

**What it asks:** Whether a claimant or their family agreed to a settlement amount and then did not return the release or respond for 30+ days.

**Question:** Did any claimant on this claim, or their family, agree to a specific settlement amount and then not return the signed release or respond for 30 or more days afterward?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when an amount was agreed, then 30+ days passed with no signed release or reply
- `No`: choose when after agreement the release came back, contact continued, or under 30 days have passed
- `No amount agreed yet`: choose when no claimant has agreed to a specific amount

### A33: Does the adjuster think the insurance money available isn't enough

**What it asks:** Whether any note says the policy limits may be too low for the injuries (an excess letter to our insured, 'damages may exceed limits').

**Question:** Does any note say the policy limits may be too low for the injuries, for example an excess letter sent to our insured or a note that damages may exceed the limits?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when an excess letter was sent, or a note says damages may exceed the limits
- `No`: choose when nothing suggests the limits may be too low

### A34: Do multiple people in the same car have oddly matching injuries and doctors

**What it asks:** Whether two or more people in the same vehicle report the same complaints and treat with the same provider on the same schedule.

**Question:** If two or more people in the same vehicle claim injury, do they report the same complaints and treat with the same provider on the same schedule, such as the same chiropractor, start date and visit days?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when two or more people in one vehicle share complaints, provider and treatment schedule
- `No`: choose when people in the same vehicle differ in complaints, provider or schedule
- `Only one person injured in that vehicle`: choose when no vehicle has two or more injured people

### A35: Does any claimant have no health insurance of their own

**What it asks:** Whether the notes say any claimant has no health insurance of their own ('no health coverage', 'self-pay').

**Question:** Do the notes say any claimant on this claim has no health insurance of their own, for example "no valid health coverage", "uninsured" or "self-pay"?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when notes say a claimant has no health coverage, is uninsured or is self-pay
- `No`: choose when health coverage is mentioned and no claimant is described as uninsured
- `Not enough facts`: choose when no claimant's health coverage is mentioned at all

### A36: Was a vehicle being used for rideshare or delivery work

**What it asks:** Whether either vehicle was doing rideshare or delivery work at the time. A routine 'ruled out Uber/Lyft' check means No.

**Question:** Was either vehicle being used for rideshare or delivery work (Uber, Lyft, DoorDash or similar) at the time of the accident? A routine "ruled out Uber/Lyft" check counts as No. If it isn't mentioned, answer No.

**Allowed values (copy exactly one, character for character):**
- `Yes, the IV`: choose when our insured's vehicle was doing rideshare or delivery work at the time
- `Yes, the OV`: choose when the other vehicle was doing rideshare or delivery work at the time
- `Both`: choose when both vehicles were doing rideshare or delivery work
- `No`: choose when neither vehicle was, or it was checked and ruled out

### A37: Does any claimant refuse the easy route and insist on going after the other driver directly

**What it asks:** Whether a claimant refused to use their own faster coverage (no-fault, med-pay, health insurance) and insisted on being paid only by the at-fault insurer.

**Question:** After being told their own coverage (no-fault, medical payments or health insurance) would pay faster, did any claimant on this claim refuse to use it and insist on being paid only by the at-fault driver's insurer?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant refused their own coverage and demanded payment only from the at-fault insurer
- `No`: choose when claimants used their own coverage, or the topic never came up

### A38: Does our own insured's story about the accident keep changing

**What it asks:** Whether our insured driver's account of the accident changes between statements or contradicts the police report.

**Question:** Does the IV driver's account of the accident change between statements, or contradict the police report, as the adjuster asks follow-up questions? Saying "I don't remember" every time counts as consistent.

**Allowed values (copy exactly one, character for character):**
- `Stays consistent`: choose when the IV driver's accounts match each other and the police report
- `Changes or contradicts`: choose when key details change between statements or conflict with the police report
- `No statement taken`: choose when no account from the IV driver is recorded

### A39: Is any claimant reporting more body parts hurt than they did at first

**What it asks:** Whether the number of body parts a claimant complains about has grown, stayed the same, or shrunk since the first report.

**Question:** Compared with the injuries first reported, does the latest information for any claimant on this claim list more body parts, the same, or fewer? If several fit, pick the highest (More, then Same, then Fewer). A claimant's first report is the first note listing that claimant's injuries; compare it with every later note listing their injuries (adjuster notes, police report, PIP application, records or a lawyer's letter). Different words for the same area (such as 'low back' and 'back') are the same body part. Use `Not enough facts` only when no claimant's injuries are listed in two different notes.

**Allowed values (copy exactly one, character for character):**
- `More`: choose when any claimant now reports a body part missing from their first report
- `Same`: choose when every claimant reports the same body parts as at first
- `Fewer`: choose when a claimant dropped body parts and no claimant added any
- `Not enough facts`: choose when body parts were reported only once, or never, for every claimant

### A40: Have the notes shifted from being about the car to being about the injury

**What it asks:** Whether the notes moved from mostly vehicle damage, rental and repairs (first third) to mostly injury, treatment and medical bills (latest third).

**Question:** Are most entries in the latest third of the notes about injury, treatment or medical bills, while most entries in the first third were about vehicle damage, rental or repairs?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when latest third mostly injury, treatment or bills, and first third mostly vehicle, rental or repairs
- `No`: choose when any other pattern, such as injury-focused from the start

### A41: Does any claimant describe real daily-life problems from the injury, not just medical terms

**What it asks:** Whether a claimant or their family described a specific daily activity the injury stops or limits (working, driving, lifting, sleeping, childcare).

**Question:** Has any claimant on this claim or their family described a specific daily activity the injury stops or limits, such as working, driving, lifting, sleeping or caring for children, rather than only naming symptoms or diagnoses?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a specific limited activity is described, such as "can't lift my child" or "missed work"
- `No`: choose when only symptoms, diagnoses or treatments are mentioned

### A42: Did any claimant's injury claim go quiet or get closed, then come back

**What it asks:** Whether a claimant's injury claim went quiet for 6+ months or was closed while they were still treating, and then came back with continuing pain or new treatment (not via a lawyer letter).

**Question:** Did any claimant's injury claim go 6+ months with no contact from their side, or get closed by us while still treating, then come back with continuing pain or new treatment? A lawyer letter or paperwork-only reopening doesn't count. If several fit, pick the first listed option after No. Silence means no incoming contact, records or bills from the claimant or their lawyer; our own unanswered attempts don't end it.

**Allowed values (copy exactly one, character for character):**
- `No`: choose when no claimant went quiet 6+ months or was closed while treating
- `Came back after we closed it`: choose when we closed a claimant's injury part while treating, then they returned with pain or treatment
- `Came back after 6+ months of silence`: choose when a claimant silent 6+ months returned with pain or new treatment
- `Quiet 6+ months, not back yet`: choose when a claimant has been silent 6+ months and has not returned

### A43: Is any claimant actively chasing payment for lost wages or help at home

**What it asks:** Whether any claimant asked for lost wages or help at home (essential services), and whether they are pursuing or disputing a denial.

**Question:** Has any claimant on this claim asked to be paid for lost wages or help at home (essential services)? If so, are they sending documents, and are they disputing a denial of these benefits? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `Not raised or declined`: choose when no claimant has asked for these, or they said they won't
- `Pursuing`: choose when a claimant has asked for them, with or without documents sent yet
- `Pursuing and disputing a denial`: choose when a claimant is pursuing and disputes a denial or reduction of these benefits

### A44: Does any claimant ask for their own copy of the police report or photos

**What it asks:** Whether a claimant or family member (not a lawyer) asked for a copy of the police report, scene photos, a recorded statement or our file.

**Question:** Has any claimant on this claim or a family member (not a lawyer) asked us for a copy of the police report, scene photos, a recorded statement or our file, other than what the repair shop needs?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant or relative asked for the police report, photos, a statement or our file
- `No`: choose when no such request, or only documents the repair shop needs

### A45: Does any claimant insist on getting things in writing

**What it asks:** Whether a claimant insisted on getting things in writing: written breakdowns, written confirmation or copies of letters. Escalating to or copying our managers is a separate feature (A46).

**Question:** Has any claimant on this claim asked for a written breakdown of a payment or offer, written confirmation of what we said, or copies of our letters or documents? Copying or escalating to our managers is not counted here (see A46).

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant asked for a written breakdown, written confirmation, or copies of letters or documents
- `No`: choose when no such requests are noted for any claimant

### A46: Has any claimant tried to escalate to our managers

**What it asks:** Whether a claimant tried to go over the adjuster's head inside our company: asked for a supervisor or manager, copied our managers or executives on emails or letters, or said they would escalate. Complaints to a state regulator are covered by A55.

**Question:** Has any claimant on this claim asked to speak with a supervisor or manager, copied our managers or executives on emails or letters, or said they would take the claim above the adjuster within our company? A complaint to a state insurance department alone counts under A55, not here.

**Allowed values (copy exactly one, word for word):**
- `Yes`: choose when a claimant asked for a supervisor or manager, cc'd our managers or executives, or said they would escalate within our company
- `No`: choose when no such escalation is noted for any claimant

### A47: Is one of the injured people a child

**What it asks:** Whether at least one injured person is under 18, from a stated age, date of birth, or 'minor'/'child'.

**Question:** Is at least one injured person on this claim under 18, based on a stated age, a date of birth, or a description such as "minor" or "child"?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any injured person is under 18 by stated age, birth date, or a word like "minor"
- `No`: choose when ages are known and every injured person is 18 or older
- `Ages not stated`: choose when some injured person's age is unknown and no one is known to be a minor

### A48: Is treatment for any claimant going on much longer than usual for this type of injury

**What it asks:** Whether any claimant's treatment has gone well past the usual length for their first reported injury (rules given in the question).

**Question:** Has the treatment of any claimant continued well past the usual length for their first reported injury? Measure from the accident to the latest treatment visit. Usual: 8 weeks for sprains, strains or bruises; 3 months for concussions or herniations without surgery; 6 months for fractures without surgery; 12 months after surgery. If only pain or soreness is reported, treat the injury as a sprain or strain (8 weeks). Measure to the latest note saying the claimant is still treating, or to the last treatment date given. Answer `No` when every claimant stopped treating or is still within the usual length. Use `Not enough facts` only when no note says whether any claimant is treating.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant is still treating beyond the usual length for their injury
- `No`: choose when every claimant finished, or is still within the usual length
- `Not enough facts`: choose when no treatment dates or injury type are noted for any claimant

### A49: Is treatment cost rising while a claimant gets harder to reach, at the same time

**What it asks:** Whether, in the latest 60 days, one claimant's medical bills rose while 3+ of our attempts to reach that same claimant went unanswered.

**Question:** In the latest 60 days of notes, did any claimant on this claim have a new medical bill, or a higher paid-to-date total, appear in the notes while three or more of our attempts to reach that same claimant went unanswered?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when in the last 60 days one claimant had a new bill or higher paid-to-date total and 3+ attempts to reach them failed
- `No`: choose when this did not happen for any claimant in the last 60 days

### A50: Does any claimant accuse our company itself of bad faith

**What it asks:** Whether a claimant or their family accused our company of lying, cheating, bad faith or unfair treatment (a lawyer's letter doesn't count).

**Question:** Has any claimant on this claim said our company cheated them, lied, acted in bad faith or treated them unfairly as a company, not just that an offer is too low or that the other driver is at fault? Count only statements by a claimant or their family; a lawyer's letter doesn't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant accuses our company of lying, cheating, bad faith or unfair treatment
- `No`: choose when only complaints about offer size or fault, or no complaints at all

### A51: Does any claimant complain it's unfair, and does any claimant say they don't want a lawyer

**What it asks:** Two things, crossed into one answer: did any claimant complain that our handling or offer is unfair, and did any claimant say they don't want or plan to hire a lawyer (with or without a complaint).

**Question:** Has any claimant on this claim said our handling or offer is unfair, and has a claimant said they don't want or plan to hire a lawyer? Decide for each claimant; if claimants differ, pick the last listed option that fits any claimant.

**Allowed values (copy exactly one, character for character):**
- `No complaint, said no lawyer wanted`: choose when no claimant called our handling or offer unfair, and a claimant said they don't want or plan to hire a lawyer
- `No complaint, said nothing about a lawyer`: choose when no claimant called our handling or offer unfair, and no claimant said they don't want a lawyer
- `Complained, said no lawyer wanted`: choose when a claimant complained and that claimant said they don't want or plan to hire a lawyer
- `Complained, said nothing about a lawyer`: choose when a claimant complained without saying they don't want a lawyer

### A52: Was the injured person outside the vehicle, like a pedestrian

**What it asks:** Where the CLAIMANT BEING SCORED was at impact: inside a vehicle, or outside as a pedestrian, bicyclist, or motorcycle or scooter rider. This one needs the specific claimant.

**Question:** Was the claimant outside an enclosed vehicle at the moment of impact, for example walking, standing, or riding a bicycle, motorcycle or scooter? If every injured person was inside a vehicle, answer `Vehicle occupant`, even when you cannot tell which one is the claimant.

**Allowed values (copy exactly one, character for character):**
- `Vehicle occupant`: choose when driver or passenger inside a car, truck, van or bus
- `Pedestrian`: choose when on foot, walking or standing, when struck
- `Bicyclist`: choose when riding a bicycle or e-bike
- `Motorcycle or scooter`: choose when riding a motorcycle, moped or scooter
- `Cannot tell`: choose when notes don't make clear which claimant this row is about, or where they were

### A53: How many times has this claim been handed off to a different person

**What it asks:** How many times the file, or a claimant's part of it, was transferred to a different adjuster or unit (temporary help doesn't count).

**Question:** How many times do the notes show the file, or any one claimant's part of it, being transferred to a different adjuster or unit? Use whichever has the most transfers. Temporary help from another adjuster doesn't count. Count each note showing the file or a claimant's part moving to a new adjuster or unit ('transferred to', 'assigned to', 'reassigned', 'bulk transfer', 'sending to' a unit). The first assignment when the claim was opened does not count, and several transfers on the same day count as one.

**Allowed values (copy exactly one, character for character):**
- `0-1`: choose when never transferred, or transferred once
- `2-3`: choose when two or three transfers
- `4+`: choose when four or more transfers

### A54: Do our own notes keep repeating the same unfinished to-do list without progress

**What it asks:** Whether three or more diary or review notes repeat the same unfinished task with nothing new done in between.

**Question:** Do three or more diary or review notes repeat the same unfinished task, such as "need medical records" or "call claimant", with nothing new done in between? Repeating the same follow-up (another request, call or reminder for the same missing item) is not progress; progress means the item arrived, was resolved or was dropped. Count notes of any kind on different dates.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when 3+ diary or review notes repeat the same open task with no progress in between
- `No`: choose when repeated tasks show progress, or no task repeats 3+ times

### A55: Did any claimant threaten to file a complaint with the state insurance regulator

**What it asks:** Whether a claimant threatened to file, or filed, a complaint with a state insurance department or consumer agency about our handling.

**Question:** Has any claimant on this claim threatened to file, or filed, a complaint about our handling with a state insurance department or consumer agency?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant threatened or filed a complaint with a state insurance department or agency
- `No`: choose when no such threat or complaint is noted

### A56: Did any claimant say they're not even sure they're making a claim

**What it asks:** Whether, within the first 30 days, a claimant said they weren't sure they would pursue an injury claim ('not sure yet', 'just want my car fixed').

**Question:** Within the first 30 days after the accident, did any claimant on this claim say they were not sure whether they would pursue an injury claim, for example "not sure yet" or "just want my car fixed"?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant said they were unsure about an injury claim or only wanted the car fixed
- `No`: choose when no claimant expressed doubt about making an injury claim

### A57: Is any claimant's injury also tied to a workers' comp claim or a health-plan lien

**What it asks:** Whether a claimant's injury is also part of a workers' comp claim, or a health plan, Medicare or Medicaid is seeking repayment (routine PIP recovery doesn't count).

**Question:** Is any claimant's injury on this claim also part of a workers' compensation claim, or is a health plan, Medicare or Medicaid seeking repayment for treating it? Routine no-fault (PIP) recovery between insurers doesn't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a workers' comp claim, or a health plan, Medicare or Medicaid lien, for any claimant
- `No`: choose when none noted, or only routine PIP recovery between insurers

### A58: Does any claimant have gig or self-employed income that's hard to document for lost wages

**What it asks:** Whether a claimant is self-employed, paid in cash, or doing gig work, so lost wages are hard to prove.

**Question:** Is any claimant on this claim self-employed, paid in cash, or doing gig work such as rideshare, delivery or contract jobs, so that lost wages are harder to prove than with a regular paycheck?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant is self-employed, paid in cash, or does gig or contract work
- `No`: choose when jobs are mentioned and none is self-employed, cash or gig work
- `Not enough facts`: choose when no claimant's job or income is mentioned

### A59: How many different body parts does the most-injured claimant say are hurt

**What it asks:** How many different body parts the most-injured claimant reports pain or injury in, as of the latest note (neck, back, shoulder, knee, head each count once).

**Question:** As of the latest note, what is the highest number of different body parts that any single claimant on this claim reports pain or injury in? Count each area, such as neck, back, shoulder, knee or head, once. Count every area ever reported for that claimant, not only in the latest note. Areas: head or face (including concussion), neck, back, chest or ribs, abdomen, shoulder or collarbone, arm or elbow, wrist or hand, hip or pelvis, leg or knee, ankle or foot. Left and right of the same area count once, and a fracture with no location named adds nothing.

**Allowed values (copy exactly one, character for character):**
- `1`: choose when the claimant with the most reports one body part
- `2`: choose when two body parts
- `3`: choose when three body parts
- `4+`: choose when four or more body parts
- `Not enough facts`: choose when no body part is named for any claimant

## Part 2: Litigation features (L01-L69)

### L01: Has any claimant's lawyer sent an evaluable medical package

**What it asks:** Whether any claimant's lawyer has sent us a usable medical package: actual medical records plus the bills for the treatment claimed. Look for 'records and bills rec'd', 'demand package', 'specials'. A letter of representation alone or a bare list of bills is not enough.

**Question:** Has any claimant's lawyer on this claim sent us a medical package we can evaluate, meaning medical records plus bills for the treatment claimed so far? A letter of representation or a bills-only list does not count.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `Package received`: choose when any claimant's lawyer sent records plus bills we can evaluate
- `No package yet`: choose when a claimant has a lawyer, but no lawyer has sent records plus bills

### L02: Time left before the deadline to sue, with no evaluable package

**What it asks:** For a claim with a lawyer but no usable medical package yet, how close the deadline to sue is. Work out the deadline from the accident date and state (state statute of limitations, often noted as 'SOL'), and measure from the latest note.

**Question:** If any claimant has a lawyer but no claimant's lawyer has sent an evaluable package (medical records plus bills), how much time is left before the state deadline to file an injury lawsuit, as of the latest note? If deadlines differ, use the earliest.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `Package received`: choose when any claimant's lawyer sent records plus bills
- `12+ months left`: choose when 12 or more months before the deadline
- `6-12 months left`: choose when 6 to under 12 months left
- `Under 6 months left`: choose when under 6 months left, deadline not yet reached
- `Deadline passed`: choose when the deadline date is already past
- `Not enough facts`: choose when accident date or state missing, so time left can't be worked out

### L03: Largest gap between a claimant's latest demand and our latest offer

**What it asks:** How far apart the two sides are in dollars: the claimant's latest demand divided by our latest offer to the same claimant, taking the claimant where that ratio is largest. Look for 'demand $', 'offer $', 'countered at', 'PL demand'.

**Question:** Across claimants with both a demand and an offer, what is the largest ratio of a claimant's latest dollar demand to our latest dollar offer to that same claimant? A 'policy limits' demand counts as a demand equal to the per-person limit when the limit is stated.

**Allowed values (copy exactly one, character for character):**
- `No demand yet`: choose when no claimant's side has asked for a dollar amount
- `No offer yet`: choose when demands exist, but no claimant has both a demand and our offer
- `Under 2x`: choose when demand is less than twice our offer
- `2x-4x`: choose when demand is 2 to under 4 times our offer
- `4x-10x`: choose when demand is 4 to 10 times our offer
- `Over 10x`: choose when demand is more than 10 times our offer
- `Not enough facts`: choose when demand and offer exist but an amount is missing, e.g. limit unstated

### L04: Are the two sides moving toward each other

**What it asks:** Whether the negotiation is converging. For the claimant with the most offers and demands traded, compare the last two demands and our last two offers and see which side moved by 10% or more toward the other.

**Question:** For the claimant whose negotiation is furthest along (most offers and demands traded), compare their last two demands and our last two offers. A side is moving if its number changed 10%+ toward the other; further apart means a demand went up or an offer went down.

**Allowed values (copy exactly one, character for character):**
- `No negotiation yet`: choose when fewer than two demands or two offers to compare
- `Both moving closer`: choose when demand and our offer each moved 10%+ toward the other
- `Only one side moving`: choose when one side moved 10%+ toward the other, the other did not
- `Neither moving`: choose when neither number moved 10% or more
- `Getting further apart`: choose when latest demand went up or our latest offer went down; pick this even if the other side moved closer
- `Not enough facts`: choose when demands and offers traded, but amounts not stated

### L05: Whose inaction caused the 60+ day quiet stretches

**What it asks:** When the file went 60+ days without real progress (no new document, offer, demand or two-way talk), who we were waiting on: us, the claimant's side, or an outside party such as a provider or police department.

**Question:** For stretches of 60+ days with no real progress on any claimant's claim (no new document, offer, demand or two-way talk, only diaries or voicemails), whose inaction mostly caused them: our company, the claimants' side, or an outside party? For each 60+ day gap, ask who owed the next step when it began; choose the side responsible for most of the total gap days.

**Allowed values (copy exactly one, character for character):**
- `No 60+ day quiet stretches`: choose when every gap between real progress is under 60 days
- `Our company`: choose when we owed the next step, e.g. unanswered package or late evaluation
- `Claimant's side`: choose when we were waiting on a claimant or their lawyer for records, demand or replies
- `Outside party such as a provider`: choose when waiting on a provider, hospital, other insurer or police report
- `Mixed`: choose when no single side caused most of the quiet time

### L06: Longest time a claimant has had a lawyer with no medical records sent to us

**What it asks:** How long the longest-represented claimant has had a lawyer without us getting any medical records, bills or demand package. Count from the letter of representation (LOR) date to the latest note.

**Question:** Across represented claimants, what is the longest time any one has had a lawyer without us receiving their medical records, medical bills or a demand package? Count from that claimant's letter of representation to the latest note.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `Records received`: choose when records, bills or a package arrived for every represented claimant
- `Under 6 months`: choose when longest wait is under 6 months
- `6-12 months`: choose when longest wait is 6 to under 12 months
- `12-18 months`: choose when longest wait is 12 to under 18 months
- `18+ months`: choose when longest wait is 18 months or more
- `Not enough facts`: choose when date of the letter of representation unknown

### L07: Does any claimant's lawyer refuse to share information

**What it asks:** How claimants' lawyers respond when we ask for information such as a recorded statement, a signed medical authorization (HIPAA) or prior providers. Use the worst response across all requests.

**Question:** When we asked any claimant's lawyer for information, such as a recorded statement, a signed medical authorization or prior providers' names, did the lawyer refuse, provide it only after 30+ days or 2+ requests, or provide it promptly? Use the worst response. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `Nothing requested yet`: choose when we have not asked any claimant's lawyer for information
- `Provided within 30 days`: choose when every request answered within 30 days on the first ask
- `Slow but never refused`: choose when a reply took 30+ days or 2+ requests, never refused
- `Refused at least once`: choose when any lawyer said no to any request

### L08: Has our company run a defense check on any claimant

**What it asks:** Whether we started a defense check on any claimant: an independent medical exam (IME), a records or peer review by our medical reviewer, or surveillance. An ISO prior-claims search does not count. Use the most advanced status reached.

**Question:** Has our company requested a defense check on any claimant, such as an independent medical exam (IME), a records or peer review by our medical reviewer, or surveillance? A routine prior-claims (ISO) search does not count. If several checks, pick the one listed last.

**Allowed values (copy exactly one, character for character):**
- `None requested`: choose when no IME, records or peer review, or surveillance on any claimant
- `Requested, lawyer refused`: choose when we asked, the lawyer refused, and no check was done
- `Done, result pending`: choose when a check was done or scheduled, result not yet in notes
- `Done, result in`: choose when the check's findings are written in the notes

### L09: What a defense check on a claimant led to

**What it asks:** What a returned defense check (IME, records or peer review, surveillance) led to: did it support the claimant, did we cut payments, treatment or our offer citing it, or did we deny the claim citing it?

**Question:** If a defense check (IME, records or peer review, surveillance) on any claimant came back, what did it lead to? Favored means it supported the injury or treatment; cut means we cut payments, treatment or our offer citing it. If several, pick denial, then cut, then favored.

**Allowed values (copy exactly one, character for character):**
- `No result yet`: choose when no defense check result is in the notes
- `Favored the claimant`: choose when a result supported a claimant's injury or treatment
- `Used to cut payments or our offer`: choose when we cut payments, treatment or our offer citing a result
- `Led to a denial of the claim`: choose when we denied a claimant's injury claim citing a result
- `No effect noted`: choose when a result came in, but notes show no action taken on it

### L10: Our company's current stance on fault between the IV and OV

**What it asks:** Our company's current liability position between our insured's vehicle (IV) and the other vehicle (OV), as stated in the latest liability note ('liab accepted 100%', '50/50', 'liab denied'). Use the BI type given in the claim facts: if it is UMBI or UIMBI, the claim is on the injured person's own policy.

**Question:** What is our company's current position on fault between the IV driver (our insured's vehicle) and the OV driver (the other vehicle), as stated in the latest liability note?

**Allowed values (copy exactly one, character for character):**
- `IV fully at fault`: choose when we accept 100% fault on the IV driver
- `Shared, IV partly at fault`: choose when we accept a partial IV share, e.g. 50/50 or 70/30
- `Disputed`: choose when we contest fault but have not denied, e.g. conflicting stories
- `Denied, OV at fault`: choose when we deny liability and place fault on the OV driver
- `Denied, stopped working file`: choose when liability denied and the injury claim is no longer being worked; pick this over Denied, OV at fault when both hold
- `Investigating`: choose when no fault decision yet, still gathering statements or reports
- `Own-policy claim`: choose when claim is on the injured person's own policy, e.g. UM or UIM (BI_TYPE UMBI or UIMBI); always pick this on own-policy claims

### L11: Longest run of unanswered attempts to reach a claimant's lawyer

**What it asks:** The current run of unanswered attempts to reach any claimant's lawyer, counted back from the latest note. Auto-replies and staff messages with no information are not answers.

**Question:** Counting back from the latest note, what is the longest current run of our attempts in a row (calls, emails, letters) to reach any one claimant's lawyer that got no real answer? An auto-reply or a staff message with no information is not a real answer.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `0`: choose when our latest attempt to each lawyer got a real answer, or no attempts
- `1-2`: choose when longest run is 1 or 2 unanswered attempts
- `3-5`: choose when longest run is 3 to 5
- `6-9`: choose when longest run is 6 to 9
- `10+`: choose when longest run is 10 or more

### L12: Has any claimant changed lawyers

**What it asks:** Whether any claimant's lawyer has changed since they first hired one: a new firm, another attorney at the same firm, or the lawyer withdrawing. Use the claimant with the most changes.

**Question:** Since each claimant first hired a lawyer, has any claimant's representation changed? A new firm, a different attorney at the same firm, or the lawyer withdrawing all count; a new paralegal or case manager does not. Use the claimant with the most changes.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has ever had a lawyer
- `No change`: choose when every represented claimant still has their first lawyer
- `New firm`: choose when one change, to a different law firm
- `New attorney, same firm`: choose when one change, to another attorney at the same firm
- `Lawyer withdrew`: choose when one change, the lawyer withdrew and no new lawyer yet
- `Two or more changes`: choose when one claimant's representation changed 2 or more times

### L13: Is there an open question about whether our policy covers this accident

**What it asks:** Whether there is a question about whether our policy covers this accident: excluded or unlisted driver, lapsed policy, business use, or a reservation-of-rights (ROR) letter, and how it was resolved.

**Question:** Is there a coverage question on our policy, such as an excluded or unlisted driver, a lapsed policy, business use, or a reservation-of-rights letter? Reversed means we accepted coverage and later reduced or denied it, or the opposite.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when notes raise no coverage question
- `Raised and cleared`: choose when a question was raised, then coverage was confirmed
- `Still open`: choose when a coverage question or reservation-of-rights letter is unresolved
- `Reduced or denied`: choose when after the question, coverage was limited or denied, with no earlier acceptance
- `Position reversed`: choose when we accepted then reduced or denied, or denied then accepted

### L14: Most serious treatment any claimant has had so far

**What it asks:** The most serious treatment documented or recommended for any claimant so far, from ER visits up to surgery, hospital admission or death. A scan only counts if it shows a fracture, herniation or tear.

**Question:** What is the most serious treatment documented or recommended for any injured claimant on this claim so far? A scan showing damage means an MRI, CT or X-ray reporting a fracture, herniation or tear, not a strain or a normal result. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None or ER only`: choose when no treatment, or only ER, urgent care or doctor visits
- `Chiro or PT only`: choose when chiropractic, physical therapy or massage, no damaging scan
- `Scan showing damage`: choose when MRI, CT or X-ray reports a fracture, herniation or tear
- `Injections`: choose when epidural, facet or trigger-point injections given or recommended
- `Surgery recommended`: choose when a doctor recommended surgery, not yet done
- `Surgery done`: choose when any surgery performed
- `Hospitalized or died`: choose when admitted overnight or longer, or a claimant died
- `Not enough facts`: choose when someone is hurt, but notes give no treatment details

### L15: Is the largest claimant injury bigger than the per-person limit

**What it asks:** Whether the most expensive claimant's injury could reach or exceed the at-fault policy's per-person limit, judged from stated medical bills and whether surgery is recommended or done. Do not use adjuster opinions or excess letters. In New York (NY) claims, medical bills are paid by no-fault (PIP) and are not part of the BI settlement, so there the bills are ignored and only surgery counts.

**Question:** Compare the highest medical bills stated for any one claimant, and whether that claimant has surgery recommended or done, with the at-fault policy's per-person limit. Don't use adjuster opinions or excess letters. If several fit, pick the highest. New York rule: if the accident state is NY, medical bills are paid by no-fault (PIP) and are not included in the BI settlement, so ignore bill amounts and judge only by surgery (no surgery = Well within the limit; surgery recommended or done = Could approach the limit; Likely exceeds the limit is not used for NY).

**Allowed values (copy exactly one, character for character):**
- `Well within the limit`: choose when largest single-claimant bills under half the limit and no surgery (NY: no surgery, bills ignored)
- `Could approach the limit`: choose when surgery recommended or done, or bills at or above half the limit (NY: surgery recommended or done)
- `Likely exceeds the limit`: choose when bills above the limit (not used for NY claims)
- `Limit not stated`: choose when the per-person limit is not given in the notes
- `Not enough facts`: choose when limit stated, but no bills or treatment details (NY: no treatment details)

### L16: Have we offered the full per-person limit to a claimant with a limit concern

**What it asks:** If a note says a claimant's damages may reach the per-person limit, whether we have offered that claimant the full per-person limit (a 'policy limits tender').

**Question:** If any note says a claimant's damages may approach or exceed the per-person limit, have we offered that claimant the full per-person limit of the at-fault policy (a policy-limits tender)? If several such claimants, use the one furthest from a full-limit offer.

**Allowed values (copy exactly one, character for character):**
- `No limit concern noted`: choose when no note says any claimant's damages may reach the limit
- `Full limit offered`: choose when we offered that claimant the full per-person limit
- `Offer below the limit`: choose when our latest offer to that claimant is below the limit
- `No offer yet`: choose when limit concern noted, but no dollar offer to that claimant

### L17: Do other parties share fault with the IV driver, and is the split still argued

**What it asks:** Whether someone besides the IV driver (the OV driver, a third driver, a road or vehicle defect) is said to share fault, and whether the percentage split is agreed or still argued.

**Question:** Besides the IV driver, is another party, such as the OV driver, a third vehicle's driver or a road or vehicle defect, said to share fault for the crash? If so, is the percentage split agreed or still argued?

**Allowed values (copy exactly one, character for character):**
- `Only the IV driver`: choose when notes put fault on no one besides the IV driver
- `Another party solely at fault`: choose when notes put no fault on the IV driver
- `Shared, split agreed`: choose when another party shares fault and the percentages are agreed
- `Shared, split still argued`: choose when another party shares fault, percentages still disputed
- `Not enough information`: choose when fault facts too thin to say whether others share it

### L18: Is any claimant incapacitated or deceased, or does a minor's payment need court approval

**What it asks:** Whether any claimant is legally incapacitated or has died, or whether a court must approve payment to a minor (for example an infant compromise). A minor whose parent is simply acting for them doesn't count. If several apply, use the one listed last.

**Question:** Is any injured claimant on this claim legally incapacitated or deceased, or do the notes say a court must approve payment to a minor (for example an infant compromise)? A minor with only a parent acting doesn't count. If several fit, pick the one listed last.

**Allowed values (copy exactly one, character for character):**
- `No`: choose when none of these noted
- `Incapacitated adult, guardian in place`: choose when a guardian or conservator is named and acting for an adult
- `Court approval needed`: choose when notes say a court must approve payment to a minor or incapacitated claimant
- `Died, estate open`: choose when a claimant died and an estate or administrator is set up
- `Died, no estate`: choose when a claimant died, no estate or administrator yet

### L19: Has the at-fault driver died

**What it asks:** Whether the at-fault driver (the IV driver, or the OV driver on an own-policy claim) has died, so the claim must go through their estate.

**Question:** Has the at-fault driver (the IV driver, or the OV driver on an own-policy claim) died, so that the claim must go through the driver's estate?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a note says the at-fault driver has died
- `No`: choose when no note says the at-fault driver died

### L20: How far the most advanced claimant's injury negotiation has gotten

**What it asks:** The furthest stage any claimant's injury claim has reached, from no claim, to a lawyer, to records, to a demand, to our offer, to counters traded back and forth.

**Question:** What is the furthest stage any claimant's injury claim on this claim has reached? Records means medical records or bills received; a demand is a dollar amount asked by a claimant's side; counters traded means 2+ offers and 2+ demands. If several fit, pick the highest. An injury claim is made when any note lists an injured person, says someone is claiming injury, opens a BI or UM injury feature for a person, or mentions a PIP application or a lawyer for an injured person. Use `No injury claim yet` only when no one is described as hurt, or everyone hurt says they won't claim.

**Allowed values (copy exactly one, character for character):**
- `No injury claim yet`: choose when no one has made an injury claim
- `Claimed, no lawyer`: choose when injury claimed, no lawyer for that claimant
- `Lawyer, no records`: choose when claimant has a lawyer, no records or bills received
- `Records, no demand`: choose when records or bills received, no dollar demand yet
- `Demand, no offer`: choose when a dollar demand made, no dollar offer from us
- `Offer made`: choose when we made at least one dollar offer
- `Counters traded`: choose when 2+ offers and 2+ demands exchanged

### L21: Does our company keep raising its offer, or has it stopped

**What it asks:** How we respond in negotiation for the claimant furthest along: after a rejection or counter, do we raise our offer by 10% or more, raise it less or not at all, or call it final while their demand is still 2x or more our offer?

**Question:** For the claimant whose negotiation is furthest along, after their side rejects or counters our offer, do we raise it 10%+, raise it under 10% or not at all, or call it our final offer while the demand is still 2x or more our offer?

**Allowed values (copy exactly one, character for character):**
- `No offer yet`: choose when we have made no dollar offer to that claimant
- `Raises 10%+ each round`: choose when each rejection or counter is followed by a raise of 10% or more
- `Raises under 10% or frozen`: choose when a raise under 10%, or no raise after a counter
- `Final offer, demand 2x+ our offer`: choose when we called it final while their demand is 2x+ our offer; pick this first when it applies
- `Not enough facts`: choose when offer made but no rejection or counter yet, or amounts missing

### L22: Has the file slowed into a holding pattern

**What it asks:** Whether the file has stalled in the latest 90 days. Compare progress events (new documents, offers, demands, two-way talks) in the latest 90 days with the 90 days before.

**Question:** In the latest 90 days of notes, has the file slowed? Progress means a new document, offer, demand or two-way talk. Holding pattern: none of these, only diaries, voicemails or 'no change'. Some slowing: progress, but longer gaps than before. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `Fewer than 5 notes`: choose when under 5 notes in the latest 90 days; this overrides the others
- `Real progress`: choose when progress with gaps no more than twice as long as in the 90 days before
- `Some slowing`: choose when some progress, but the longest gap between progress events is at least twice the longest gap in the 90 days before
- `Holding pattern`: choose when no progress in 90 days, only diaries, voicemails or 'no change'

### L23: Is the claimant's current lawyer the same one from a past claim

**What it asks:** From prior-claims searches (ISO or similar), whether the lawyer now representing the claimant being scored also represented them on an earlier, separate injury claim. Needs the claimant named in the claim facts.

**Question:** From prior-claims searches in the notes (ISO or similar), did the lawyer now representing this claimant also represent them on an earlier, separate injury claim? If no prior-claims search is noted, or no claimant has a lawyer now, answer `No prior represented claim`, also when you cannot tell which injured person is the claimant.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a search shows the same lawyer or firm on an earlier injury claim
- `No`: choose when an earlier represented claim exists, with a different lawyer
- `No prior represented claim`: choose when no search, or no earlier claim where they had a lawyer
- `Cannot tell`: choose when several claimants, and notes don't show which one this row is

### L24: Is our company disputing that any claimant's injury is real or from this accident

**What it asks:** Whether our notes take a firm position that any claimant's injury is fake, unrelated to this accident, pre-existing, or below the state's serious-injury bar. Only asking for prior records is a weaker step and not a firm position.

**Question:** Do our notes take the position that any claimant's injuries are not real, not caused by this accident, pre-existing, or below the state's serious-injury bar? Only asking for prior records is not a firm position. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No dispute`: choose when notes accept every claimant's injuries or are silent
- `Only asked for prior records`: choose when we asked for prior records or claims history, no stated position
- `Firm position in the notes`: choose when a note says an injury is fake, unrelated, pre-existing or below the bar

### L25: Have we told the claimants' side we dispute an injury

**What it asks:** If our notes firmly dispute a claimant's injury, whether we actually told that claimant or their lawyer, in a letter, a call or when explaining an offer.

**Question:** If our notes take a firm position that any claimant's injury is not real, unrelated to this accident or pre-existing, have we told that claimant or their lawyer so in a letter, a call or an offer explanation?

**Allowed values (copy exactly one, character for character):**
- `No firm dispute`: choose when notes take no firm position against any claimant's injury
- `Dispute not shared`: choose when firm position in notes, never told to that claimant's side
- `Told the claimant's side`: choose when we told the disputed claimant or their lawyer, by letter, call or offer

### L26: Did the IV driver do something a jury would punish, apart from impairment

**What it asks:** How bad the IV driver's conduct was in a way a jury would punish, leaving out alcohol or drug impairment. Ranges from ordinary carelessness up to fleeing, racing or a criminal charge.

**Question:** Leaving aside alcohol or drug impairment (asked separately), did the IV driver do something that would anger a jury? If several fit, pick the highest. Use what the police report, a citation, a witness, our insured's own admission or our liability decision shows. One driver's accusation alone counts only as `Ordinary carelessness`. If we decided the OV driver was fully at fault and no IV error is described, answer `None noted`.

**Allowed values (copy exactly one, character for character):**
- `None noted`: choose when notes describe no driving error by the IV driver
- `Ordinary carelessness`: examples: rear-ending, failing to yield, unsafe lane change
- `Serious`: choose when texting or phone use, running a red light or stop sign, or 20+ mph over the limit
- `Egregious`: choose when fleeing the scene, racing, or a criminal charge from the crash other than DUI/DWI

### L27: Has mediation or arbitration been proposed, and by whom

**What it asks:** Whether mediation, a settlement conference or arbitration was proposed for any claimant's injury claim, and by which side. No-fault (PIP) arbitration over a denied bill does not count.

**Question:** Has mediation, a settlement conference or arbitration been proposed for any claimant's injury claim, and by which side? A general 'call me to discuss' is not a proposal. Don't count no-fault (PIP) arbitration over a denied treatment or bill.

**Allowed values (copy exactly one, character for character):**
- `None proposed`: choose when no such proposal for any claimant
- `Our company proposed`: choose when only our side proposed it
- `Claimant's side proposed`: choose when only a claimant or their lawyer proposed it
- `Both sides proposed`: choose when each side proposed it at some point

### L28: What happened to the mediation or arbitration proposal

**What it asks:** The latest status of a mediation, settlement-conference or arbitration proposal: declined, waiting for an answer, agreed or scheduled, or already held.

**Question:** If mediation, a settlement conference or arbitration was proposed for any claimant's claim, what is its latest status? Declined means the other side said no or did not answer for 30+ days. If several proposals, use the one listed last.

**Allowed values (copy exactly one, character for character):**
- `None proposed`: choose when no such proposal for any claimant
- `Declined`: choose when the other side said no, or gave no answer for 30+ days
- `No answer yet, under 30 days`: choose when proposed under 30 days ago, no reply yet
- `Agreed or scheduled`: choose when both sides agreed, or a date is set
- `Held`: choose when the session took place; ignore its outcome

### L29: Is any claimant out of step with their own lawyer

**What it asks:** Whether any claimant is out of step with their own lawyer: the lawyer can't reach them, the lawyer dropped them, or the claimant contacts us directly wanting to settle faster. If several, use the one listed last.

**Question:** Do the notes show any claimant out of step with their own lawyer: the lawyer says they can't reach the client, the lawyer dropped the client, or the claimant contacts us directly wanting to resolve the claim faster? If several fit, pick the one listed last.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `No sign of a problem`: choose when no friction noted between any claimant and their lawyer
- `Lawyer can't reach client`: choose when a lawyer says they lost contact with their client
- `Lawyer dropped client`: choose when a lawyer withdrew citing the client, e.g. non-cooperation
- `Claimant pushing to resolve`: choose when a represented claimant contacts us directly to resolve faster

### L30: Does paying any claimant depend on another insurer that hasn't committed

**What it asks:** Whether paying a claimant depends on another insurer or policy (a second at-fault vehicle's insurer, an umbrella, or consent from the claimant's own underinsured coverage) whose share is not yet confirmed in writing.

**Question:** Does paying any claimant depend on another insurer or policy, such as a second at-fault vehicle's insurer, an umbrella policy, or consent from a claimant's own underinsured coverage, whose share is not yet confirmed in writing?

**Allowed values (copy exactly one, character for character):**
- `No other insurer needed`: choose when our policy alone can pay every claimant
- `Needed, share confirmed`: choose when every needed share or consent is confirmed in writing
- `Needed, still open`: choose when any needed share or consent is not yet confirmed in writing

### L31: Has any claimant's lawyer demanded the full policy limit

**What it asks:** Whether any claimant's lawyer demanded the full per-person policy limit, either as a dollar amount equal to the limit or in words such as 'policy limits demand' or 'PL demand'.

**Question:** Has any claimant's lawyer demanded the full per-person policy limit, either as a dollar figure equal to the limit or in words such as 'policy limits demand'?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any claimant's lawyer demanded the per-person limit, by amount or in words
- `No`: choose when a claimant has a lawyer, but none demanded the full limit
- `Not represented`: choose when no claimant on this claim has a lawyer

### L32: How serious the documented treatment was when the limit was demanded

**What it asks:** If a lawyer demanded the full policy limit, how serious the treatment was in that claimant's records at the moment of the demand. This tests whether the limits demand was backed by serious documented injury.

**Question:** If any claimant's lawyer demanded the full policy limit, what was the most serious treatment in that claimant's records we had at that time? Serious means surgery, injections, a hospital stay, or a scan showing a fracture or tear. If several demands, use the most serious.

**Allowed values (copy exactly one, character for character):**
- `No limits demand`: choose when no claimant's lawyer demanded the full limit
- `Serious treatment documented`: choose when records then showed surgery, injections, a hospital stay or damaging scan
- `Only chiro, PT or ER`: choose when records then showed only chiro, PT, ER or doctor visits
- `No records sent yet`: choose when we had no medical records for that claimant then
- `Not enough facts`: choose when demand date or when records arrived is unclear

### L33: Who is paying the medical bills of the claimant with the highest bills

**What it asks:** Who is paying the medical bills of the claimant with the highest stated bills (or the most serious treatment if no bills are stated): no-fault (PIP or med pay) and how much of it is used, health insurance, workers' comp, or no one.

**Question:** Who is currently paying medical bills for the claimant with the highest stated medical bills (if none are stated, the one with the most serious treatment)? Use the share of their no-fault (PIP) or medical-payments limit already paid. If several payers pay, pick the first that applies: workers' comp, no-fault, health insurance, no one. If a note shows PIP or med-pay is the primary payer, or a PIP or med-pay claim is open for that claimant, and no exhaustion or denial is noted, answer `No-fault, under half used`, even if nothing has been paid yet. Use `Not enough facts` only when no note mentions any payer or benefit claim.

**Allowed values (copy exactly one, character for character):**
- `No-fault, under half used`: choose when PIP or med-pay paying, under half the limit paid
- `No-fault, over half used`: choose when PIP or med-pay paying, half or more of the limit paid
- `No-fault used up`: choose when limit exhausted, or an exhaustion notice sent
- `No-fault denied`: choose when PIP or med-pay denied or cut off entirely
- `Health insurance only`: choose when only a health plan, Medicare or Medicaid is paying
- `Workers' compensation`: choose when a workers' comp carrier is paying
- `No one`: choose when bills unpaid, or the claimant pays out of pocket
- `Not enough facts`: choose when notes don't say who is paying

### L34: Is someone else owed money out of any claimant's recovery

**What it asks:** Whether a health plan, Medicare, Medicaid, a hospital or another party has claimed a lien on any claimant's recovery, and whether the amount is confirmed in writing. Routine PIP recovery between insurers doesn't count.

**Question:** Has a health plan, Medicare, Medicaid, a hospital or another party claimed a lien on any claimant's recovery, and is the amount confirmed in writing? Routine no-fault (PIP) recovery between insurers does not count. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None known`: choose when no lien noted on any claimant's recovery
- `One lien, amount confirmed`: choose when a single lien with a final amount in writing
- `Amount unknown or disputed`: choose when a lien exists, amount missing, conditional or disputed
- `Several liens or a government lien`: choose when 2+ liens across claimants, or any Medicare or Medicaid lien

### L35: On an own-policy claim, where does the OV driver's insurance stand

**What it asks:** On a claim under the injured person's own uninsured or underinsured motorist coverage (BI type UMBI or UIMBI), where things stand with the at-fault OV driver's insurance. On a regular BI liability claim, answer `Not an own-policy claim`.

**Question:** If this is a claim on an injured person's own uninsured or underinsured motorist coverage, what is the status of the OV (at-fault vehicle) insurance? Limit offered means the OV insurer offered its full limit.

**Allowed values (copy exactly one, character for character):**
- `OV uninsured or unknown`: choose when OV had no insurance, hit-and-run, or insurer not identified
- `OV limit offered`: choose when the OV insurer offered its full limit
- `OV limit not confirmed`: choose when OV insurer known, but limit or offer not confirmed
- `Others competing for OV limit`: choose when other injured people are claiming the same OV limit; pick this whenever others claim it, regardless of offer status
- `OV claim not started`: choose when no claim made yet against the OV insurer
- `Not an own-policy claim`: choose when this is a liability claim against our insured

### L36: Is our insured cooperating with us

**What it asks:** How cooperative our insured or the IV driver is with us: answers promptly, slow, unreachable, or refusing a statement, documents or contact. Use the worst behavior seen at any time.

**Question:** How cooperative is our insured or the IV driver with us? Reporting the loss or giving an account of the accident (including at the first report) counts as a reply. Slow: they replied at some point, but at some time 3 or more of our attempts in a row went unanswered, or a reply took 14+ days; this includes going silent after earlier contact. Unreachable: they never replied and never gave an account, despite at least one attempt. Not cooperating: they refused a statement, documents or contact, or told us to stop contacting them. If behavior varied, use the worst: Not cooperating, Unreachable, Slow, Cooperative.

**Allowed values (copy exactly one, character for character):**
- `Cooperative`: choose when they reported the loss or answered us, every reply came within 14 days, and never 3+ attempts in a row went unanswered
- `Slow but reached`: choose when they replied at some point, but at some time 3+ attempts in a row went unanswered or a reply took 14+ days, including going silent after earlier contact
- `Unreachable`: choose when they never replied or gave an account, despite at least one attempt
- `Not cooperating`: choose when they refused a statement, documents or contact, or told us to stop contacting them
- `Not enough facts`: choose when no note shows any contact or attempt with our insured or the IV driver, including at the first report

### L37: Has our insured hired their own lawyer or brought in another insurer

**What it asks:** Whether our insured or the IV driver hired a personal lawyer of their own (not defense counsel we hired) or reported the accident to an excess or umbrella insurer.

**Question:** Has our insured or the IV driver hired a personal lawyer of their own, or reported the accident to an excess or umbrella insurer? A lawyer our company hires for them does not count. Count a lawyer representing our insured or the IV driver for any reason, including their own injury claim, as `Own lawyer` (look for a letter of representation for the insured, or 'insd has atty'). On a UMBI or UIMBI claim our insured is the claimant, so their injury lawyer does not count here.

**Allowed values (copy exactly one, character for character):**
- `No`: choose when no personal lawyer and no excess or umbrella insurer noted
- `Own lawyer`: choose when insured or IV driver hired a personal lawyer, not one we hired
- `Excess or umbrella insurer`: choose when accident reported to an excess or umbrella insurer
- `Both`: choose when a personal lawyer and an excess or umbrella insurer

### L38: Is paperwork, not money, holding up payment to any claimant

**What it asks:** Whether paperwork we need before paying any claimant is still outstanding, such as a Medicare eligibility form, a W-9 tax ID form or guardianship documents, and for how long.

**Question:** Are paperwork items we need before paying any claimant still outstanding, such as a Medicare eligibility form, a tax ID form or a guardianship document? Count it as dragging if 60+ days outstanding or the lawyer refuses. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None outstanding`: choose when no needed payment paperwork pending for any claimant
- `Outstanding under 60 days, lawyer cooperating`: choose when requested under 60 days ago, no refusal
- `Outstanding 60+ days or lawyer refusing`: choose when pending 60+ days, or a lawyer or claimant refused

### L39: Days from an evaluable package to our first offer

**What it asks:** For the claimant furthest along in negotiation, how quickly we made a first dollar offer after receiving their usable medical package (records plus bills), or how long we have gone without offering.

**Question:** For the claimant whose negotiation is furthest along, how many days passed between our receiving their evaluable package (medical records plus bills) and our first dollar offer to them?

**Allowed values (copy exactly one, character for character):**
- `No package yet`: choose when no evaluable package from any claimant
- `Under 30 days`: choose when first offer under 30 days after the package
- `30-90 days`: choose when first offer 30 to 90 days after the package
- `Over 90 days`: choose when first offer more than 90 days after the package
- `No offer yet, under 60 days`: choose when no offer, package arrived under 60 days ago
- `No offer after 60+ days`: choose when no offer, package arrived 60+ days ago
- `Not enough facts`: choose when package or offer date not stated

### L40: Time left before the deadline to sue when our first offer arrived

**What it asks:** How much time was left before the state's deadline to sue on the date of our first dollar offer to any claimant. If no offer yet, measure from the latest note.

**Question:** How much time was left before the state deadline to file an injury lawsuit on the date of our first dollar offer to any claimant on this claim? If there is no offer yet, use the time left as of the latest note.

**Allowed values (copy exactly one, character for character):**
- `No offer, 12+ months left`: choose when no offer yet, 12+ months to the deadline now
- `No offer, under 12 months left`: choose when no offer yet, under 12 months to the deadline now
- `12+ months`: choose when first offer made with 12+ months left
- `6-12 months`: choose when first offer made with 6 to under 12 months left
- `Under 6 months`: choose when first offer made with under 6 months left
- `Not enough facts`: choose when accident date, state or offer date unknown

### L41: When a lawyer asked for the policy limits, did we tell them

**What it asks:** If any claimant's lawyer asked us to disclose the policy limits, how we responded: told them within 30 days, late or with conditions, or refused.

**Question:** If any claimant's lawyer asked us to disclose the policy limits, how did we respond? Promptly means within 30 days with no conditions; conditions means we first required records, a statement or an affidavit. If several requests, use the worst response.

**Allowed values (copy exactly one, character for character):**
- `Not asked`: choose when no lawyer asked for the policy limits
- `Told within 30 days`: choose when limits disclosed within 30 days, no conditions
- `Told late or with conditions`: choose when disclosed after 30+ days, or only after records, statement or affidavit
- `Refused`: choose when we declined to disclose, no withholding by the lawyer noted after
- `Refused, lawyer now withholding info`: choose when we declined, then the lawyer held back records or info
- `Not enough facts`: choose when asked, but notes don't show our response

### L42: What reason we gave for an offer under half the demand

**What it asks:** For the claimant with the largest demand-to-offer gap, if our latest offer is under half their demand, what reason we gave: simple arithmetic both sides accept (prior payments, PIP offset), a dispute over injury, treatment or cost, or no reason. Policy-limit demands are ignored: lawyers often demand the full limit, especially on soft-tissue claims, and it does not show the claim's true value. Only real dollar demands below the limit are used.

**Question:** For the claimant with the largest demand-to-offer gap, if our latest offer is under half their demand, what reason did we give? Arithmetic means deductions both sides accept, such as prior payments or a no-fault offset. Disputed means we question the injury, treatment or cost. Ignore policy-limit demands: a demand for "policy limits" or "PL", or a dollar demand equal to or above the per-person limit, is not a real valuation (lawyers make it often, especially on soft-tissue claims such as sprains, strains or whiplash). For each claimant use their latest demand below the limit, and find the largest gap using only those demands.

**Allowed values (copy exactly one, character for character):**
- `No offer yet`: choose when no dollar offer to any claimant with a demand (counting policy-limit demands)
- `Only policy-limit demands so far`: choose when every demand so far is a policy-limit demand, so there is no real demand to compare our offer with
- `Offer half the demand or more`: choose when our latest offer is at least half the demand
- `Arithmetic only`: choose when only agreed deductions, e.g. prior payments or PIP offset
- `Disputed injury, treatment or cost`: choose when we said injury, treatment or bills are excessive or unrelated; pick this whenever any dispute is given
- `No reason given`: choose when notes show no explanation given to the claimant's side

### L43: Is our dispute of a claimant's injury backed by a formal review

**What it asks:** If we told a claimant's side our offer is low because we dispute the injury, treatment or bills, whether a formal review backs that up (IME, records or peer review, bill review) or it is just the adjuster's opinion.

**Question:** If we told any claimant's side our offer is low because we dispute the injury, treatment or bills, is that backed by a formal review such as an IME, a medical records or peer review, or a bill-review report?

**Allowed values (copy exactly one, character for character):**
- `No dispute given`: choose when we never gave any claimant's side a dispute as the reason
- `Backed by a formal review`: choose when an IME, records or peer review, or bill review supports it
- `Adjuster's opinion only`: choose when any stated dispute has no formal review behind it

### L44: How many treatments the claimants' own insurers have denied

**What it asks:** If any claimant gets no-fault (PIP) or medical-payments benefits under their own policy, how many treatments or bills were denied, or whether benefits were cut off. Use the claimant with the most denials.

**Question:** If any claimant gets no-fault (PIP) or medical-payments benefits under their own policy, how many treatments or bills have been denied, or have benefits been cut off? Use the claimant with the most denials. A request for more paperwork is not a denial. If several fit, pick the highest. A claimant gets benefits when notes show a PIP or med-pay claim or coverage for them from any source (their own policy, a household policy, our policy or a state fund such as PLIGA), even if nothing has been paid yet. Use `Not applicable (no own-policy benefits)` only when no such claim or coverage is noted for any claimant.

**Allowed values (copy exactly one, character for character):**
- `Not applicable (no own-policy benefits)`: choose when no claimant gets PIP or med-pay benefits
- `None`: choose when benefits paid, no denials noted
- `1-2`: choose when one or two treatments or bills denied
- `3+ or benefits cut off`: choose when three or more denials, or benefits stopped, e.g. after an IME

### L45: Are several injured people competing for the same limited money

**What it asks:** Whether several injured people are claiming against the same at-fault policy, and whether the money is tight compared with the per-accident limit.

**Question:** Are several injured people on this claim claiming against the same at-fault policy? Money is tight if their combined demands, or injuries such as surgeries, likely exceed the per-accident limit.

**Allowed values (copy exactly one, character for character):**
- `Only injured person`: choose when only one person is claiming injury under the policy
- `Several, enough money`: choose when 2+ injured, combined demands or injuries likely under the limit
- `Several, money tight`: choose when 2+ injured and combined dollar demands exceed the per-accident limit, or 2+ have surgery recommended or done
- `Others already paid from the limit, less left`: choose when some injured people already paid, less remains; pick this whenever anyone was already paid from the limit

### L46: Did we talk with the first represented claimant before a lawyer got involved

**What it asks:** For the first claimant to hire a lawyer, whether our adjuster had real two-way talks with them about the injury, treatment or settlement before the lawyer appeared, and how much.

**Question:** For the first claimant on this claim to hire a lawyer, did the adjuster have a two-way talk with them about the injury, treatment or settlement beforehand? Weeks of talks means 2+ conversations over 14+ days; brief means any shorter contact. A two-way talk is a live call or meeting with the claimant (or a family member speaking for them) about the injury, treatment or settlement; the talk in which the claimant first tells us they have a lawyer does not count as before the lawyer. Measure the 14 days from the first to the last such talk before the lawyer. If the first note about this claimant's injury came from a lawyer, answer `Lawyer reported claim`. If no claimant has a lawyer, answer `Still no lawyer, in contact` when we had any two-way talk with a claimant (including one saying they are not hurt), otherwise `Not enough facts`.

**Allowed values (copy exactly one, character for character):**
- `Still no lawyer, in contact`: choose when no claimant has a lawyer, and we have talked with one
- `Weeks of talks, then lawyer`: choose when 2+ talks over 14+ days before the lawyer came
- `Brief talk, then lawyer`: choose when any two-way talk before the lawyer that doesn't meet Weeks of talks
- `Never reached, then lawyer`: choose when no two-way talk before the lawyer came
- `Lawyer reported claim`: choose when a lawyer's letter was our first notice of the injury claim
- `Not enough facts`: choose when no lawyer and no contact yet, or timing unclear

### L47: Claimant's relationship to our insured

**What it asks:** The relationship of the claimant being scored (named in the claim facts) to our named insured or the IV driver: stranger, non-family passenger, family or household, friend or coworker, or our own policyholder.

**Question:** What is this claimant's relationship to our named insured or the IV driver? Family or household means a relative or anyone living with them; own policyholder means the claimant is our named insured. If the notes say which vehicle the claimant was in and give no tie to our insured or the IV driver (for example an OV driver or OV passenger), answer `Stranger`. Use `Not enough facts` only when nothing says who the claimant is or which vehicle they were in.

**Allowed values (copy exactly one, character for character):**
- `Stranger`: choose when no tie to the insured or IV driver, e.g. OV driver
- `Non-family passenger`: choose when rode in the IV, not family, friend or coworker
- `Family or household`: choose when a relative of, or lives with, the insured or IV driver
- `Friend or coworker`: choose when a friend or coworker of the insured or IV driver
- `Own policyholder`: choose when the claimant is our named insured
- `Not enough facts`: choose when relationship not stated in the notes
- `Cannot tell`: choose when several claimants, and notes don't show which one this row is

### L48: Did our side drop something the claimants' side sent

**What it asks:** Whether our side dropped something from the claimants' side: a lawyer's letter, demand or package left unacknowledged 14+ days, misfiled, sent to the wrong adjuster, or a lawyer complaining we never answered.

**Question:** Do the notes show a slip on our side, such as a lawyer's letter, demand or medical package left unacknowledged for 14+ days, misfiled, or sent to the wrong adjuster, or a lawyer complaining we never responded? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No slip`: choose when nothing from any claimant's side missed, misfiled or left unanswered
- `Slip fixed within 30 days`: choose when a slip happened and we responded within 30 days
- `Not fixed or fixed 30+ days late`: choose when still unanswered, or fixed 30+ days after it arrived

### L49: Does the serious-injury bar to sue apply, and can a claimant clear it

**What it asks:** Whether a serious-injury bar (a rule that allows a lawsuit for pain and suffering only above an injury threshold) applies to this claim, and whether the claimant most likely to meet it can do so. Decide with the state rules in the question, not with other knowledge.

**Question:** Does a serious-injury bar to suing for pain and suffering apply here, and can the claimant most likely to meet it do so? Use the accident state and only these rules. NY, MI, FL: the bar applies. MA: the bar applies, and it is met by medical bills over $2,000, a fracture, or permanent disfigurement or loss. NJ: the bar applies when the notes show the limitation-on-lawsuit (verbal threshold, "LOL") option, does not apply when they show the no-limitation (zero threshold) option, and is unknown when the option is not stated. PA: the bar applies when the notes show limited tort, does not apply with full tort, and is unknown when not stated. CT, NH and every other state: no bar. Likely met means a fracture, surgery or a scan showing damage (in MA also medical bills over $2,000). When the bar applies and no note shows any of these, answer `Applies, no objective findings`, even when injury details are thin.

**Allowed values (copy exactly one, character for character):**
- `Doesn't apply`: choose when the rules above give no bar for this state or tort option
- `Applies, likely met`: choose when the bar applies and a claimant has a fracture, surgery or damaging scan (in MA also medical bills over $2,000)
- `Applies, we dispute it`: choose when a claimant says the bar is met, our notes dispute it; pick this whenever we dispute the bar, even with a fracture, surgery or damaging scan
- `Applies, no objective findings`: choose when the bar applies and no note shows a fracture, surgery or damaging scan (in MA, nor medical bills over $2,000), including when injury details are thin
- `Not eligible to sue`: choose when notes say no claimant can sue, e.g. limited tort, no qualifying injury
- `Not enough facts`: choose when the accident is in NJ or PA and the notes don't state the tort option, or the accident state is unknown

### L50: Did any claimant hire a lawyer right after a direct offer or payment

**What it asks:** Whether any claimant hired a lawyer within 30 days after we made them a direct injury offer or payment while they had no lawyer, such as a quick-settle offer or a check for medical bills.

**Question:** Did any claimant hire a lawyer within 30 days after we made them, while they had no lawyer, an injury offer or payment, such as a quick-settle offer or a check for medical bills?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a claimant got a lawyer within 30 days of our direct offer or payment
- `No`: choose when we made a direct offer or payment, and no lawyer followed within 30 days
- `Not applicable (no lawyer, or lawyer before any offer)`: choose when no direct offer or payment came before any claimant's lawyer

### L51: Was the file handed off at a critical moment

**What it asks:** Whether the file moved to a new adjuster at a critical time (within 30 days after a demand, or within 6 months of the deadline to sue), and whether the new adjuster contacted the claimants' side within 14 days. Vacation coverage doesn't count.

**Question:** Was this claim's file moved to a new adjuster within 30 days after any claimant's demand or within 6 months of the deadline to sue? If so, did the new adjuster contact the claimants' side within 14 days? Temporary coverage doesn't count.

**Allowed values (copy exactly one, character for character):**
- `No critical handoff`: choose when no reassignment in those windows, or only vacation cover
- `Handed off, contact within 14 days`: choose when new adjuster contacted the claimants' side within 14 days
- `Handed off, no contact within 14 days`: choose when new adjuster made no contact within 14 days

### L52: Has any claimant's treatment escalated since our last evaluation without us catching up

**What it asks:** Whether any claimant's treatment escalated after our latest evaluation or offer (new specialist, scan showing damage, injections, surgery), and whether we re-evaluated within 30 days of learning about it.

**Question:** Since our latest evaluation or offer for each claimant, has any claimant's treatment escalated (new specialist, scan showing damage, injections, or surgery recommended or done)? If so, did we re-evaluate within 30 days of learning of it? If several fit, use the one listed last.

**Allowed values (copy exactly one, character for character):**
- `No evaluation yet`: choose when we have not evaluated or made an offer for any claimant
- `No escalation`: choose when no claimant's treatment escalated after our latest evaluation
- `Escalated, re-evaluated within 30 days`: choose when we re-evaluated within 30 days of learning of it
- `Escalated, not re-evaluated`: choose when no re-evaluation within 30 days of learning of it

### L53: Did our company take the first step toward settling

**What it asks:** The most our company did on its own initiative to move any claimant's claim toward settlement, from only asking for records up to inviting a settlement meeting or sending a written evaluation with an offer.

**Question:** What is the most our company has done, without being asked, to move any claimant's claim toward settlement? Structured push means inviting the lawyer to a settlement meeting or roundtable, or sending a written evaluation with an offer. If several fit, pick the highest. Asking for records, bills, a statement, a medical authorization or a PIP application, or taking a statement, counts as `Only asked for records`. An internal evaluation, liability decision or reserve review with nothing sent to the claimant's side is `Evaluated, not shared`.

**Allowed values (copy exactly one, character for character):**
- `Only asked for records`: choose when we only requested records, statements or bills
- `Evaluated, not shared`: choose when we evaluated internally but shared nothing
- `Made an offer`: choose when we made a dollar offer on our own initiative
- `Structured push`: choose when invited a settlement meeting, or sent a written evaluation with an offer
- `Not enough facts`: choose when no injury claim has been made yet, or we have taken no action at all on it

### L54: What extra damages beyond medical bills are claimed

**What it asks:** Beyond medical bills, the biggest extra damages item claimed for any claimant: short-term lost wages, extended time off work, lost job or earning ability, or scarring or psychological injury. If several, use the one listed last.

**Question:** Beyond medical bills, what is the biggest extra damages item claimed for any claimant on this claim? Short-term means under 4 weeks off work; extended means 4+ weeks; lost earning ability means a lost job or a permanent work restriction. If several fit, pick the one listed last. Being out of work ('OOW') or filing for disability or wage benefits counts as lost wages. Use `None` when a claimant's injury claim or treatment is described but no lost wages, time off work, job loss, scarring or psychological injury is mentioned. Use `Not enough facts` only when no claimant has made an injury claim yet.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when only medical bills are claimed
- `Short-term lost wages`: choose when under 4 weeks off work claimed
- `Extended time off work`: choose when 4 or more weeks off work claimed
- `Lost job or earning ability`: choose when a lost job or permanent work restriction claimed
- `Scarring or psychological injury`: choose when scarring, disfigurement, PTSD, anxiety or depression claimed
- `Not enough facts`: choose when notes don't say what damages are claimed

### L55: Is our offer 30% or more below our own documented value

**What it asks:** Whether our latest offer to any claimant is 30% or more below the value we ourselves documented for that claimant, such as an evaluation note or approved authority (use the bottom of a range).

**Question:** Is our latest offer to any claimant 30% or more below the settlement value we documented ourselves for that claimant, such as an evaluation note or approved authority? If we documented a range, use its bottom.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when for any claimant, our latest offer is 30%+ below our documented value
- `No`: choose when every offer is within 30% of our documented value
- `No offer yet`: choose when no dollar offer to any claimant
- `No value documented`: choose when offers exist, but no evaluation or authority amount in notes

### L56: What injury information we have actually gotten

**What it asks:** The best injury and treatment information we have actually obtained for any claimant, from nothing at all up to actual medical records or bills that we reviewed.

**Question:** What is the best injury and treatment information we have actually obtained for any claimant? Records means actual medical records or bills, not a summary from someone. Records reviewed is highest, Nothing is lowest. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `Records reviewed`: choose when we received and reviewed actual medical records or bills
- `No-fault payment figures only`: choose when only PIP or med-pay payment amounts
- `Lawyer's verbal updates`: choose when only a lawyer's or their staff's description of treatment
- `Claimant's word only`: choose when only what a claimant told us
- `Records ordered, not in`: choose when we requested records, none received yet
- `Nothing`: choose when no injury information obtained at all

### L57: How the lead claimant's lawyer is approaching settlement

**What it asks:** For the claimant furthest along in negotiation, their lawyer's stance on settling based on the latest contact. Ignore any statements about filing suit. Apply the tie-break order in the question.

**Question:** For the claimant whose negotiation is furthest along, what is their lawyer's stance on settling, based on the latest contact? Engaging: discusses numbers or counters. Rigid: refuses to move off the demand or to negotiate at all. Ignore any statements about filing suit. If several fit, pick the first that applies: Recommending our offer, Rigid, Engaging, Neutral. Real contact means any two-way exchange with the lawyer's office (the attorney or staff), including a call they made to us; a letter of representation alone or our unanswered messages are not contact.

**Allowed values (copy exactly one, character for character):**
- `No contact yet`: choose when no lawyer, or no real contact with that lawyer yet
- `Engaging`: choose when the lawyer discusses numbers or makes counters
- `Recommending our offer to client`: choose when the lawyer says they will recommend or present our offer
- `Neutral, noncommittal`: choose when the lawyer responds but takes no position
- `Rigid`: choose when the lawyer refuses to move off the demand or negotiate

### L58: Has the adjuster ever talked live with a claimant's actual attorney

**What it asks:** The deepest contact our adjuster has had with any claimant's lawyer's office: voicemails or letters only, office staff only, a live talk with the attorney, or a live talk with the attorney trading dollar figures.

**Question:** What is the deepest contact the adjuster has had with any claimant's lawyer's office? Talking with the attorney means a live call or meeting with the attorney, not a paralegal or case manager. If several fit, pick the highest. Count talking with the attorney only when a note says one of our staff spoke live with the attorney (named, 'spoke to/with atty', or an incoming call from the attorney), including at the first report. A call to 'the PA office', or a talk with a secretary, paralegal, case manager or unnamed office person, is office staff.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `Voicemails or letters only`: choose when only one-way messages, no live conversation
- `Office staff only`: choose when live talks only with paralegals, case managers or staff
- `Talked with attorney about injuries`: choose when live call or meeting with an attorney, no dollar figures
- `Traded numbers with attorney`: choose when discussed demand or offer amounts live with an attorney

### L59: Did any claimant go ahead with treatment after it was denied

**What it asks:** After an insurer denied or cut off a treatment for any claimant, whether the claimant or their doctor went ahead with it anyway, said they would, or dropped it.

**Question:** After any insurer denied or cut off a treatment for any claimant, did that claimant or their doctor go ahead with it anyway, for example surgery done after a denial or PT continuing after a cut-off? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No denials`: choose when no insurer denied or cut off any claimant's treatment
- `Treatment dropped`: choose when the treatment stopped after the denial
- `Said they'll go ahead`: choose when claimant or doctor says they will proceed anyway
- `Went ahead anyway`: choose when treatment was done or continued after the denial
- `Not enough facts`: choose when denial noted, but notes don't say what happened next

### L60: Is any insurance company willing to pay a claimant

**What it asks:** Whether at least one possible payer (the OV insurer, our coverage, or a claimant's own policy) is accepting any claimant's injury claim by accepting fault or coverage or making an offer, or whether all have denied.

**Question:** Is at least one possible payer, such as the OV insurer, our coverage, or a claimant's own policy, accepting any claimant's injury claim on this claim, meaning it accepted fault or coverage or made an offer?

**Allowed values (copy exactly one, character for character):**
- `At least one accepting`: choose when a payer accepted fault or coverage, or made an offer, for any claimant
- `None yet, some undecided`: choose when no acceptance yet, at least one payer still deciding
- `All have denied`: choose when every possible payer denied every claimant's injury claim

### L61: Is our own insured pushing back against a claimant

**What it asks:** Whether our insured, the IV driver or their family actively dispute any claimant's account, such as who caused the crash or whether the claimant is really hurt, and how strongly.

**Question:** Does our insured, the IV driver or their family actively dispute any claimant's account, such as who caused the crash or whether the claimant is really hurt? Active means raised 2+ times or backed with photos, witnesses or a statement. If several fit, pick the highest. Count only when our insured's side contradicts something a claimant said or claimed (about fault or injury). A recorded or written statement that contradicts a claimant counts as backed. Our insured simply giving their own account, with no claimant account to contradict, is `No`.

**Allowed values (copy exactly one, character for character):**
- `No`: choose when notes show no disagreement from our insured's side
- `Disagreed once`: choose when disputed a claimant's account once, without evidence
- `Active, repeated dispute`: choose when raised 2+ times, or backed by photos, witnesses or a statement

### L62: Has our view of fault shifted between the IV and OV

**What it asks:** Comparing our earliest and latest written fault assessments, whether fault has moved toward our insured's vehicle (IV), toward the other vehicle (OV), or stayed the same.

**Question:** Comparing our earliest and latest written fault assessments, has fault moved toward the IV (our insured's vehicle), toward the OV (the other vehicle), or stayed the same?

**Allowed values (copy exactly one, character for character):**
- `Only one assessment`: choose when fewer than two written fault assessments
- `Moved toward the IV`: choose when the later assessment puts more fault on our insured's driver
- `Unchanged`: choose when the fault split is the same in both
- `Moved toward the OV`: choose when the later assessment puts more fault on the other driver

### L63: Has any claimant's lawyer sent a time-limited demand

**What it asks:** Whether any claimant's lawyer sent a time-limited demand: one that must be accepted by a stated date or within a set number of days (for example 'open for 30 days').

**Question:** Has any claimant's lawyer sent a demand that must be accepted by a stated date or within a set number of days, for example 'this offer to settle is open for 30 days'?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any claimant's lawyer sent a demand with an acceptance deadline
- `No`: choose when a claimant has a lawyer, but no demand had an acceptance deadline
- `Not represented`: choose when no claimant on this claim has a lawyer

### L64: Has any claimant's lawyer sent a bad-faith letter

**What it asks:** Whether any claimant's lawyer sent a formal letter accusing us of bad faith or unfair claim practices, or citing a state unfair-claims law. Merely complaining an offer is too low doesn't count.

**Question:** Has any claimant's lawyer sent a formal letter accusing our company of bad faith or unfair claim practices, or citing a state unfair-claims law? Only complaining that an offer is too low does not count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any claimant's lawyer sent a letter alleging bad faith or unfair practices
- `No`: choose when a claimant has a lawyer, but no such letter; low-offer complaints only
- `Not represented`: choose when no claimant on this claim has a lawyer

### L65: Did any claimant's treatment start late or only after a lawyer referral

**What it asks:** Whether any claimant's first treatment started 14+ days after the accident, or only after a lawyer sent them to a provider. An ER or urgent-care visit within 14 days counts as starting on time.

**Question:** Did any claimant's first treatment start 14+ days after the accident, or only after a lawyer sent them to a provider? An ER or urgent-care visit within 14 days counts as an on-time start.

**Allowed values (copy exactly one, character for character):**
- `Too early to tell`: choose when under 14 days since the accident and no claimant treated yet
- `Started on time, own choice`: choose when every treated claimant started within 14 days, not via a lawyer
- `Started 14+ days late or after lawyer referral`: choose when any claimant first treated 14+ days later, or via a lawyer
- `Not enough facts`: choose when notes give no treatment start dates or referral source

### L66: Is any claimant's treatment far heavier than the crash facts suggest

**What it asks:** Whether treatment looks far heavier than the crash: a minor crash (drivable, no airbags, no tow, no ambulance) while a claimant had injections or surgery recommended or done, or 6+ months of therapy.

**Question:** Was the crash minor (vehicle drivable, no airbags, no tow, no ambulance) while any claimant still had surgery or injections recommended or done, or 6+ months of therapy? Decide `No` first: if the crash was not minor (any tow, airbag, ambulance, or a car that could not be driven), the answer is `No` whatever the treatment. Use `Too early to tell` only for a minor crash when the latest treatment is under 2 months after the accident. Use `Not enough facts` only when the notes describe neither the crash damage nor any treatment.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when minor crash and any claimant had surgery or injections recommended or done, or 6+ months of therapy
- `No`: choose when the crash was not minor, or treatment stayed below that level
- `Too early to tell`: choose when treatment just started, under 2 months in
- `Not enough facts`: choose when crash or treatment details missing from notes

### L67: Have the claimants' lawyers asked for our file documents

**What it asks:** Apart from the policy limits, how many different file documents claimants' lawyers asked us for: a certified policy copy, a recorded statement, photos, the police report and so on. Count distinct documents across all lawyers.

**Question:** Apart from the policy limits, how many different file documents, such as a certified copy of the policy, a recorded statement, photos or the police report, have claimants' lawyers asked us for? Count distinct documents in total across all lawyers.

**Allowed values (copy exactly one, character for character):**
- `Not represented`: choose when no claimant on this claim has a lawyer
- `0`: choose when no lawyer asked for any file document
- `1`: choose when one distinct document requested in total
- `2+`: choose when two or more distinct documents requested in total

### L68: Is any claimant or claimant's lawyer from another state

**What it asks:** Whether any claimant lives in, or any claimant's lawyer practices in, a different state from where the accident happened.

**Question:** Do the notes show that any claimant lives in, or any claimant's lawyer practices in, a different state from the one where the accident happened? Use any location clue: a home or office address or city, the state of a claimant's own insurer, license plate or driver's license, or a phone area code. Answer `Same state` unless a note points to an out-of-state claimant or lawyer; silence about locations means `Same state`.

**Allowed values (copy exactly one, character for character):**
- `Same state`: choose when no note points to an out-of-state claimant or lawyer, including when locations are not mentioned
- `Claimant from another state`: choose when a note points to a claimant living out of state, and no lawyer out of state
- `Lawyer from another state`: choose when a note points to a lawyer practicing out of state, and no claimant out of state
- `Both`: choose when an out-of-state claimant and an out-of-state lawyer
- `Not enough facts`: choose when no claimant has been identified in the notes yet

### L69: Do the notes debate which state's rules apply

**What it asks:** Whether the notes actually debate which state's law or insurance rules govern the claim, such as whose no-fault or serious-injury rules apply. Boilerplate naming one state doesn't count.

**Question:** Do the notes discuss which state's law or insurance rules govern the claim, such as which state's no-fault or serious-injury rules apply? Boilerplate naming a single state's rules does not count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when notes discuss which state's law, no-fault or serious-injury rules apply
- `No`: choose when no such discussion, or only boilerplate naming one state

## Part 3: Severity features (S01-S43)

### S01: Most serious objectively proven injury in the first 90 days

**What it asks:** Using only notes from the first 90 days after the accident, the most serious injury that is objectively confirmed (by imaging, hospital records or a surgeon) for any claimant. Pain complaints alone are the lowest level; a bulge or degenerative change is not a positive scan.

**Question:** Using only notes from the first 90 days, what is the most serious injury confirmed by imaging, hospital records or a surgeon among all injured claimants on this claim? Positive scan means a herniation or tear on MRI/CT/EMG, not a bulge or degenerative change. If several fit, pick the highest. Use only notes dated within the first 90 days. An ER or hospital visit with no diagnosis or test result noted is `Pain only`. Treat a dislocation like a fracture. Use `Not enough facts` only when the first 90 days of notes mention no injury complaint at all.

**Allowed values (copy exactly one, character for character):**
- `Pain only`: choose when pain complaints, but no exam finding or test confirms an injury
- `Soft tissue, normal tests`: choose when sprain or strain diagnosed; scans normal or show only bulges or degeneration
- `Positive scan`: choose when MRI, CT or EMG shows a herniation, tear or nerve finding
- `Fracture or surgical tear`: choose when any fracture, or a tear a surgeon says needs repair
- `Brain bleed, admission or death`: choose when brain bleed, admitted overnight, or anyone died
- `Not enough facts`: choose when notes from the first 90 days describe no diagnosis, test or care

### S02: Did the claimants stop pursuing the claim in the first 180 days

**What it asks:** Within the first 180 days after the accident, how many claimants stopped pursuing their injury claim. Signs: 3+ unanswered contacts in a row, forms requested twice and never returned, two missed medical exams, saying they won't claim, or their lawyer losing contact.

**Question:** In the first 180 days, how many of the injured claimants stopped pursuing their injury claim? Signs of stopping: 3+ unanswered calls or letters in a row, forms requested twice and never returned, two missed medical exams, saying they won't claim, or their lawyer losing contact. Count each unanswered call, voicemail, text or contact letter as one attempt. Count only people described as injured or making a claim; a person who tells us they were not hurt or will not claim shows a sign of stopping.

**Allowed values (copy exactly one, character for character):**
- `All pursuing`: choose when no claimant shows any sign of stopping
- `Some stopped`: choose when at least one claimant shows a sign while another is still pursuing
- `All stopped`: choose when every claimant shows a sign or says they won't claim

### S03: Per-person limit of the coverage that would pay the claimant

**What it asks:** The per-person limit of the coverage on the scored claimant's row, from the first note that states coverage. Use the BI type given in the claim facts: for BI, the IV bodily-injury liability limit; for UMBI or UIMBI, the claimant's own UM/UIM limit. Split limits like '25/50' mean $25,000 per person. Never use the PIP limit.

**Question:** From the first coverage note, what is the per-person limit of the coverage on this claimant's row: the IV bodily-injury liability limit for BI, or the claimant's own UM/UIM limit for UMBI/UIMBI? Never use the PIP limit. Ignore an umbrella unless confirmed. On a BI claim the IV's per-person limit is the same for every claimant, so a stated BI limit answers this even when you cannot tell which injured person is the claimant.

**Allowed values (copy exactly one, character for character):**
- `$15,000 or less`: examples: 15/30
- `$15,001-$50,000`: examples: 25/50 or 50/100
- `$50,001-$249,999`: examples: 100/300
- `$250,000 or more`: examples: 250/500, or a single limit of $250,000+
- `Cannot tell`: choose when limit not stated, or unclear which claimant applies

### S04: Room left on the claimant's own UM/UIM coverage

**What it asks:** If the scored claimant claims under their own uninsured or underinsured motorist coverage (BI type UMBI or UIMBI), how much more their own per-person limit is than the at-fault driver's per-person limit, after any step-down or policy that pays first. An uninsured or hit-and-run driver counts as a $0 at-fault limit.

**Question:** If the claimant claims under their own UM/UIM coverage, how much does its per-person limit exceed the at-fault driver's per-person limit, after any step-down or policy that pays first? An uninsured or hit-and-run driver means the full limit. On a claim whose BI type is BI, the answer is `Not applicable`.

**Allowed values (copy exactly one, character for character):**
- `Not applicable`: choose when the claimant makes no claim under their own UM/UIM
- `None, at-fault limit as high or higher`: choose when own limit not above the at-fault driver's limit
- `Under $25,000`: choose when own limit exceeds the at-fault limit by less than $25,000
- `$25,000-$99,999`: choose when own limit exceeds the at-fault limit by $25,000-$99,999
- `$100,000 or more`: choose when own limit exceeds the at-fault limit by $100,000 or more
- `Cannot tell`: choose when either limit unknown, or unclear which claimant's policy

### S05: Biggest medical bill figure known by day 180

**What it asks:** The largest dollar figure for any one claimant's accident medical charges stated within 180 days of the accident: a hospital or ER bill, a PIP paid-to-date total, or a lien amount. Never use a demand, offer, reserve or settlement number.

**Question:** What is the largest dollar figure stated within 180 days for any one injured claimant's own accident medical charges on this claim, such as a hospital or ER bill, a PIP paid-to-date total or a lien amount? Never use a demand, offer, reserve or settlement figure. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None reported`: choose when no medical dollar figure stated within 180 days
- `Under $5,000`: choose when largest figure is below $5,000
- `$5,000-$24,999`: choose when largest figure is $5,000 to $24,999
- `$25,000-$99,999`: choose when largest figure is $25,000 to $99,999
- `$100,000 or more`: choose when largest figure is $100,000 or more

### S06: How fast treatment moved to injections or surgery (done or recommended)

**What it asks:** How many days after the accident a treating provider first gave or formally recommended an injection, nerve block or surgery for any claimant. Use the earliest such date and real note dates. A surgical recommendation counts the same as surgery done, even if the surgery has not happened yet or was later denied.

**Question:** Using real note dates, how many days after the accident did a treating provider first give, or formally recommend, an injection, nerve block or surgery for any claimant on this claim? Use the earliest such date across all claimants. Include surgical recommendations: if a doctor or surgeon recommended surgery, count it from the date it was first recommended, whether or not the surgery was done.

**Allowed values (copy exactly one, character for character):**
- `Under 60 days`: choose when first given or recommended 0-59 days after the accident
- `60-120 days`: choose when first given or recommended 60-120 days after
- `121-180 days`: choose when first given or recommended 121-180 days after
- `Over 180 days`: choose when first given or recommended more than 180 days after
- `None so far`: choose when no injection, block or surgery given or recommended for anyone
- `Date unknown`: choose when one was given or recommended, but its date can't be placed

### S07: New injections or surgery after a prior same-body-part problem

**What it asks:** If any claimant had a prior problem with the same body part (for example an old back injury), whether care after this accident stayed conservative (therapy, chiropractor, medication) or moved to injections or surgery never done on that body part before.

**Question:** If any claimant on this claim had a prior problem with the same body part, did that claimant's care after this accident stay conservative (therapy, chiropractor, medication) or move to injections or surgery not done on that body part before? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No prior problem`: choose when no claimant has a documented prior same-body-part problem
- `Prior, care stayed conservative`: choose when prior problem; care since is only therapy, chiropractor or medication
- `Prior, then new injections or surgery`: choose when prior problem; then injections or surgery new to that body part
- `Not enough facts`: choose when prior problem known, but care after this accident isn't described

### S08: Size of the first medical lien

**What it asks:** The size of the first stated lien amount from a health plan, Medicare, Medicaid, workers' comp or a hospital on each claimant's recovery, taking the largest across claimants. Ignore car-loan lienholders, child-support liens and PIP recovery between insurers.

**Question:** Taking each claimant's first stated dollar amount for a health plan, Medicare, Medicaid, workers' compensation or hospital lien on their recovery, what is the largest on this claim? Ignore car-loan lienholders, child-support liens and PIP recovery between insurers. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No lien`: choose when no qualifying lien asserted for any claimant
- `Lien with no amount`: choose when a qualifying lien exists, but no dollar amount stated yet
- `Under $5,000`: choose when largest first lien amount below $5,000
- `$5,000-$24,999`: choose when largest first lien amount $5,000 to $24,999
- `$25,000 or more`: choose when largest first lien amount $25,000 or more

### S09: Most serious head or brain injury

**What it asks:** The most serious head or brain injury documented for any claimant, from a head strike or headache up to a brain bleed or skull fracture. A head strike is split into alleged (only the claimant's word) and objective (physical proof, such as a spiderwebbed windshield). 'LOC' means loss of consciousness; 'TBI' means traumatic brain injury.

**Question:** What is the most serious head or brain injury documented among all injured claimants on this claim? Lasting symptoms means a doctor notes post-concussion symptoms beyond about 60 days, or a neurology or neuropsychology referral. If several fit, pick the highest. Physical proof of a head strike means something others can see: a spiderwebbed or starred windshield, a dented headliner or pillar, or a cut, bump or bruise on the head noted by EMS or the ER. The claimant saying they hit their head is not proof.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no head complaint or head injury noted for anyone
- `Head strike alleged or headache only`: choose when the claimant says they hit their head, or has a headache, with no physical proof of a head strike and no concussion diagnosis
- `Head strike with physical proof`: choose when a head strike backed by physical evidence, such as a spiderwebbed or starred windshield, a dented headliner or pillar, or a cut, bump or bruise on the head seen by EMS or the ER; no concussion diagnosis
- `Concussion or knocked out`: choose when concussion diagnosed, or any loss of consciousness
- `Lasting symptoms or neurologist`: choose when post-concussion symptoms past about 60 days, or neurology or neuropsychology referral
- `Brain bleed or skull fracture`: choose when bleeding or bruising on brain imaging, or a skull fracture

### S10: Visible scarring or disfigurement

**What it asks:** Whether any claimant has visible scarring or disfigurement documented or alleged: a stitched cut, scar photos, a plastic-surgeon referral, a visible bump from a healed fracture. Facial scarring is the highest level.

**Question:** Is visible scarring or disfigurement of any claimant on this claim documented or alleged, such as a stitched cut, scar photos, a plastic-surgeon referral or a visible bump from a healed fracture? Lost teeth alone go no higher than the body-scar level. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no scar or disfigurement mentioned for anyone
- `Alleged only`: choose when claimant or lawyer claims scarring, nothing documented
- `Stitched cut or scar on the body`: choose when stitches, staples, scar photos or lost teeth, but nothing on the face
- `Facial scarring, deformity or reconstructive surgery`: choose when scar on the face, visible deformity, or plastic or reconstructive surgery

### S11: Most this coverage can actually pay after sharing and offsets

**What it asks:** The most the coverage on the scored claimant's row (BI, or own UM/UIM per the BI type in the claim facts) can actually pay that claimant after pro-rata sharing, a guest-passenger step-down, excess position behind another policy, aggregate already used by others, or a small first-party cap.

**Question:** After pro-rata sharing with another policy, a guest-passenger step-down, excess position behind a rental or owner's policy, aggregate already used by others, or a small first-party cap, what is the most the coverage on this claimant's row (BI, or own UM/UIM per BI_TYPE) can pay the claimant? If the notes mention no sharing, step-down, excess position, used aggregate or cap, the answer is the band of the per-person limit itself. On a BI claim that limit is the same for every claimant.

**Allowed values (copy exactly one, character for character):**
- `Under $5,000`: choose when less than $5,000 can be paid, e.g. a small first-party cap
- `$5,000-$24,999`: choose when $5,000 to $24,999 can be paid after offsets
- `$25,000-$99,999`: choose when $25,000 to $99,999 can be paid after offsets
- `$100,000 or more`: choose when $100,000 or more can be paid after offsets
- `Cannot tell`: choose when limit or offsets unknown, or unclear which claimant

### S12: Was follow-up care finished or still going at day 90 (longest-treating claimant)

**What it asks:** Using notes up to about day 90 after the accident, whether the longest-treating claimant's follow-up care after the ER was already finished or still going at day 90. Any source counts: PIP payer, lawyer, bills or the claimant.

**Question:** Using notes up to about day 90 after the accident, from any source (PIP payer, lawyer, bills or a claimant), was follow-up care after the ER for the longest-treating claimant already finished, or still going at day 90? If today's date is before day 90, judge at the latest note: still treating then counts as `Still treating at day 90`. A scene check or ER visit with no later care mentioned is `No care beyond the ER`. Use `Not enough facts` only when the notes up to day 90 mention no care at all for anyone.

**Allowed values (copy exactly one, character for character):**
- `No care beyond the ER`: choose when only an ER or single visit, no follow-up care
- `Short course finished`: choose when under 8 weeks of care, ended by day 90
- `Still treating at day 90`: choose when therapy, chiropractic or specialist care ongoing at day 90
- `Not enough facts`: choose when notes up to day 90 don't describe anyone's treatment

### S13: Claimant's age

**What it asks:** The scored claimant's age on the accident date, from a stated age or date of birth ('DOB'). If the only age given belongs to someone else, answer `Cannot tell`.

**Question:** How old was the claimant on the accident date, from a stated age or date of birth? If the only age given belongs to another occupant, answer Cannot tell.

**Allowed values (copy exactly one, character for character):**
- `Under 18`: choose when age 0-17 on the accident date
- `18-39`: choose when age 18-39 on the accident date
- `40-64`: choose when age 40-64 on the accident date
- `65-79`: choose when age 65-79 on the accident date
- `80 or older`: choose when age 80+ on the accident date
- `Cannot tell`: choose when no age or birth date for this claimant, or unclear whose age is given

### S14: Highest level of hospital care in the first 14 days

**What it asks:** The highest level of hospital care any claimant received in the first 14 days after the accident, from none, through urgent care and the ER, to admission, ICU or surgery during a stay.

**Question:** What was the highest level of hospital care any injured claimant on this claim received in the first 14 days after the accident? Outpatient surgery means a same-day procedure under anesthesia, such as a wound or eye repair. If several fit, pick the highest. Count care in the first 14 days even when a later note reports it; if a care date isn't given, use the date of the note that first mentions it. 'Taken to', 'transported to' or 'went to' a hospital counts as an ER visit, and if no admission is mentioned it is `ER, treated and released`. Use `None` when the notes say no one was hurt or everyone declined care. Use `Not enough facts` only when someone was hurt and no note says whether they got any care in those 14 days.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no medical care for anyone in the first 14 days
- `Urgent care or doctor visit`: choose when urgent care or own doctor, no ER
- `ER, treated and released`: choose when ER visit, sent home the same day
- `Outpatient surgery`: choose when same-day procedure under anesthesia, not admitted
- `Admitted overnight`: choose when admitted to hospital for at least one night
- `ICU or surgery during stay`: choose when ICU care, or surgery while admitted
- `Not enough facts`: choose when notes from the first 14 days don't say whether anyone got care

### S15: Was recommended surgery actually done

**What it asks:** Within 365 days of the accident, whether a treating surgeon recommended surgery for any claimant and what happened: performed, pending, declined by the claimant, or denied by a reviewer and not done. Apply the tie-break order in the question.

**Question:** Within 365 days, has a treating surgeon recommended surgery for any claimant on this claim, and was it performed (even after a pre-certification denial), still pending, declined, or denied by a reviewer and not done? If claimants differ, pick the first that fits: Performed, pending, denied, declined.

**Allowed values (copy exactly one, character for character):**
- `None recommended`: choose when no surgeon recommended surgery for anyone within 365 days
- `Recommended, pending`: choose when recommended, not yet done and not refused or denied
- `Declined or not done`: choose when the claimant refused or chose not to have it
- `Denied by reviewer, not done`: choose when insurer or utilization review denied it, and it wasn't done
- `Performed`: choose when the surgery was done, even after a denial

### S16: Most invasive procedure recommended for any claimant

**What it asks:** The most invasive procedure a treating doctor recommended, requested or scheduled for any claimant's accident injuries, even if denied or not done yet. Ranges from injections to fusion, joint replacement or implanted devices.

**Question:** What is the most invasive procedure a treating doctor has recommended, requested or scheduled for any injured claimant's accident injuries on this claim, even if denied or not yet done? Manipulation under anesthesia counts as Injections only. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no procedure recommended for anyone
- `Injections only`: choose when epidural, facet, trigger-point or joint injection, nerve block, ablation, or manipulation under anesthesia
- `Arthroscopic joint surgery`: choose when scope of a knee, shoulder, hip, wrist or other joint
- `Disc surgery or fracture fixation`: choose when discectomy, laminectomy, plates, screws or rods for a fracture, or other surgery under general anesthesia such as abdominal, brain or facial reconstruction
- `Fusion, joint replacement or implanted device`: choose when spinal fusion, joint or disc replacement, spinal cord stimulator or pump

### S17: Most invasive procedure performed on any claimant

**What it asks:** The most invasive procedure actually performed on any claimant for accident injuries, shown by an operative report, a bill or a note that it was done. Recommended or scheduled only doesn't count here.

**Question:** What is the most invasive procedure actually performed on any injured claimant on this claim for accident injuries, shown by an operative report, a bill or a note that it was done? Recommended or scheduled only doesn't count. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no procedure performed on anyone
- `Injections only`: choose when epidural, facet, trigger-point or joint injection, nerve block, ablation, or manipulation under anesthesia
- `Arthroscopic joint surgery`: choose when scope of a knee, shoulder, hip, wrist or other joint
- `Disc surgery or fracture fixation`: choose when discectomy, laminectomy, plates, screws or rods for a fracture, or other surgery under general anesthesia such as abdominal, brain or facial reconstruction
- `Fusion, joint replacement or implanted device`: choose when spinal fusion, joint or disc replacement, spinal cord stimulator or pump

### S18: How and when injured claimants first got medical care

**What it asks:** How and when injured claimants first got medical care: an ambulance from the scene, an ER on the same day, care 1-14 days later, care after 14 days, or no care. Pick the first option in the list that fits any claimant.

**Question:** Among all injured claimants on this claim, what was the most urgent first medical care after the accident, and how many days later? Ambulance counts only if a claimant rode in it; leaving the ER against medical advice counts as refused. If several fit, pick the first option that fits. Count `Ambulance from the scene` only when the notes say ambulance, EMS or 'transported' from the scene; 'taken to the hospital' without that is `Same-day ER on own`. If the first care date isn't stated, use the date of the note that first mentions that care.

**Allowed values (copy exactly one, character for character):**
- `Ambulance from the scene`: choose when any claimant rode an ambulance from the scene
- `Same-day ER on own`: choose when any claimant reached an ER on the accident day without an ambulance
- `Within 1-14 days`: choose when earliest care for anyone was 1 to 14 days later
- `More than 14 days later`: choose when earliest care for anyone was over 14 days later
- `Refused or no care`: choose when every injured claimant refused care, left against advice, or got none
- `Not enough facts`: choose when notes don't say how or when anyone first got care

### S19: Was the claimant belted and kept in place

**What it asks:** If the scored claimant was inside a vehicle, whether they were belted and stayed in place, or were unbelted, ejected, or struck the windshield, steering wheel or a broken seat back. Pedestrians, cyclists and motorcyclists are not occupants.

**Question:** If the claimant was inside a vehicle at impact, were they belted and kept in place, or were they unbelted, ejected, or did they strike the windshield, steering wheel or a broken seat back?

**Allowed values (copy exactly one, character for character):**
- `Not an occupant`: choose when pedestrian, cyclist, motorcyclist or scooter rider
- `Restrained occupant`: choose when belted, not ejected, no listed interior strike
- `Unbelted, ejected or hit interior`: choose when unbelted, thrown out, or struck the windshield, wheel or a broken seat back
- `Cannot tell`: choose when belt use not stated, or unclear which claimant

### S20: Gaps in treatment or missed appointments

**What it asks:** Within the first 180 days after the accident, whether any claimant had a gap of more than 30 days between treatment visits, or missed or cancelled 3 or more appointments. Use real note dates.

**Question:** Within the first 180 days, using real note dates, did any claimant on this claim have a gap of more than 30 days between treatment visits, or miss or cancel 3 or more appointments?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any claimant had an over-30-day treatment gap or 3+ missed or cancelled visits
- `No`: choose when everyone who treated did so without such gaps or misses
- `Not applicable (no treatment)`: choose when no claimant had any treatment beyond a single visit

### S21: Type of future care put forward for any claimant

**What it asks:** Whether anyone has put forward future care beyond the current treatment for any claimant: more injections or therapy, a future surgery (including repeat surgery or hardware removal), or lifelong care or a permanency opinion.

**Question:** Has anyone put forward future care beyond current treatment for any injured claimant on this claim? Future surgery includes repeat surgery and hardware removal; lifelong care includes lifelong procedures or a permanency or future-arthritis opinion. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no future care mentioned for anyone
- `Repeat injections or ongoing therapy`: choose when future injections, therapy or medication beyond the current course
- `Future surgery`: choose when surgery, repeat surgery or hardware removal expected later
- `Lifelong care or permanency opinion`: choose when lifelong care or procedures, or a permanency or future-arthritis opinion

### S22: Strongest support for future care

**What it asks:** If future surgery, repeat procedures or lifelong care was put forward for any claimant, who backs it most strongly: only the claimant or their lawyer, a treating doctor, or our own medical review.

**Question:** If future surgery, repeat procedures or lifelong care was put forward for any claimant on this claim, who supports it most strongly? Our medical review counts only if our exam or nurse case manager agreed it is accident-related. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No future care put forward`: choose when no such future care proposed for anyone
- `Claimant or lawyer only`: choose when only a claimant, lawyer or their hired expert asserts it
- `Treating doctor recommends`: choose when a treating doctor recommends it in records or notes
- `Our own medical review agrees`: choose when our exam or nurse case manager agrees it is accident-related
- `Not enough facts`: choose when future care proposed, but who supports it isn't stated

### S23: Treatment escalated despite a low-impact flag

**What it asks:** If our company flagged the crash as low impact or doubted that it caused the injury in the first 90 days, whether any claimant still got injections or surgery (done or formally recommended) within 180 days of the accident.

**Question:** If the company flagged the crash as low impact or doubted causation in the first 90 days, did any claimant on this claim still get injections or surgery, performed or formally recommended, within 180 days of the accident?

**Allowed values (copy exactly one, character for character):**
- `No flag raised`: choose when no low-impact or causation flag in the first 90 days
- `Flagged, care stayed conservative`: choose when flagged; no claimant got or was recommended injections or surgery by day 180
- `Flagged, injections or surgery followed`: choose when flagged; any claimant got or was recommended injections or surgery by day 180

### S24: Is only reimbursement being sought

**What it asks:** The broadest kind of claim made for any claimant within the first 180 days: nothing, only another insurer seeking repayment, only out-of-pocket costs (deductibles, co-pays, short lost wages), or a pain-and-suffering claim.

**Question:** In the first 180 days, what is the broadest kind of claim made for any claimant on this claim: nothing, another insurer's reimbursement, out-of-pocket costs such as deductibles, co-pays or lost wages for under 2 weeks, or a pain-and-suffering claim? If several fit, pick the highest. Count a `Pain-and-suffering claim` when a claimant or their lawyer reports an injury claim against our insured or under UM/UIM, or a BI or UM injury feature is opened for an injured claimant, unless the notes say the claimant wants only specific out-of-pocket costs. Use `No claim pursued` only when no injured person has made any claim.

**Allowed values (copy exactly one, character for character):**
- `No claim pursued`: choose when no injury-related claim made for anyone within 180 days
- `Another insurer's reimbursement only`: choose when only a health plan or other carrier seeks repayment
- `Out-of-pocket costs only`: choose when deductibles, co-pays, lost wages for under 2 weeks, no pain and suffering
- `Pain-and-suffering claim`: choose when any claimant or their lawyer seeks pain-and-suffering or general injury damages

### S25: Another insurer already offered its full limit

**What it asks:** Within the first 180 days, whether another insurer that could owe a claimant bodily-injury damages (such as another at-fault vehicle's carrier or a primary rental-car policy) offered or paid its full per-person limit to any claimant. Our own payments don't count.

**Question:** Within the first 180 days, has another insurer that could owe a claimant bodily-injury damages offered or paid its full per-person limit to any claimant on this claim? Count only these other insurers: the liability carrier of another at-fault vehicle or driver; a primary liability policy such as a rental-car or employer policy; and, on a UIMBI claim, the at-fault driver's carrier. Do not count a claimant's own no-fault (PIP), med-pay or health insurance, or a carrier handling only vehicle damage. Our own payments don't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when one of those other insurers offered or paid its full per-person limit to any claimant
- `No`: choose when one of those other insurers is involved (named or described in the notes, or the claim is UIMBI), but no full-limit offer or payment yet
- `Not applicable (no other insurer)`: choose when no note shows such an insurer; also when the only other carriers are no-fault, med-pay, health or vehicle-damage carriers

### S26: Type of provider treating in the first 90 days

**What it asks:** In the first 90 days, the highest-level provider treating any claimant after the ER: primary care or PT, chiropractor or pain clinic, an independent specialist, or a hospital surgical team.

**Question:** In the first 90 days, what is the highest-level provider treating any injured claimant on this claim after the ER? Count a provider when a note says a claimant is treating with, seeing, or going to them (for example "treating with chiro" or "in PT 3x/wk"), unless the notes say there was only one visit. A clinic doing its own MRIs and trigger-point injections is a pain clinic; an independent specialist is an outside orthopedist, neurologist or physiatrist. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No one beyond the ER`: choose when a claimant went to the ER or a hospital and no note names any later provider in the first 90 days
- `Primary care or physical therapy`: choose when family doctor, urgent-care follow-up or physical therapist
- `Chiropractor or pain clinic`: choose when chiropractor, or a clinic doing its own MRIs and trigger-point injections
- `Independent specialist`: choose when outside orthopedist, neurologist or physiatrist
- `Hospital surgical team`: choose when hospital surgeons or trauma team manage care
- `Not enough facts`: choose when no note mentions any medical care (ER, hospital or provider) for any claimant in the first 90 days

### S27: First real demand (below the limit) compared with the limit

**What it asks:** The first specific dollar demand below the policy limit made for any claimant, as a share of that claimant's per-person limit. Policy-limit demands are skipped: lawyers demand the full limit on nearly every claim regardless of severity, so it says nothing about value. Ignore requests to disclose limits. For a global demand covering several claimants, divide by the combined limit.

**Question:** What was the first specific dollar demand below the policy limit made for any claimant on this claim, as a share of that claimant's per-person limit? Skip policy-limit demands: a demand for "policy limits" or "PL", or a dollar demand equal to or above the limit, is not counted, because lawyers make it on nearly every claim regardless of severity. Ignore limits-disclosure requests; for a global demand, divide by the combined limit. If claimants differ, use the highest share.

**Allowed values (copy exactly one, character for character):**
- `No demand yet`: choose when no specific dollar demand made for anyone
- `Only policy-limit demands so far`: choose when every demand so far was for policy limits, or at or above the limit
- `Under 25%`: choose when first demand below the limit is under a quarter of the limit
- `25%-74%`: choose when first demand below the limit is from a quarter to just under three quarters of the limit
- `75%-99%`: choose when first demand below the limit is from three quarters of the limit to just under the limit
- `Not enough facts`: choose when a dollar demand was made, but the limit is unknown, so it can't be placed

### S28: Rehab, nursing or home care after the hospital

**What it asks:** Within 180 days of the accident, whether any claimant needed care beyond the hospital: inpatient rehab, a nursing facility, a visiting nurse or home health aide, or home changes like a ramp, stair lift or wheelchair.

**Question:** Within 180 days, did any claimant on this claim need care beyond the hospital: inpatient rehab, a nursing facility, a visiting nurse or home health aide, or home changes such as a ramp, stair lift or wheelchair? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no post-hospital care or home changes for anyone
- `Home health aide or home changes`: choose when visiting nurse, home aide, ramp, stair lift, wheelchair or similar
- `Inpatient rehab or nursing facility`: choose when any stay in a rehab hospital or nursing facility

### S29: Doctor-documented permanent loss

**What it asks:** Whether a treating doctor (not a lawyer or hired expert) documented a specific permanent loss for any claimant, such as lost vision or hearing, an amputation, permanent hardware or a cognitive deficit. A soft-tissue permanency form doesn't count.

**Question:** Has a treating doctor, not a lawyer or hired expert, documented a specific permanent loss for any claimant on this claim, such as lost vision or hearing, amputation, permanent hardware or a cognitive deficit? A soft-tissue permanency form doesn't count.

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when a treating doctor documents a specific permanent loss for any claimant
- `No`: choose when no such documentation; lawyer claims or soft-tissue permanency forms only count as No

### S30: Psychological treatment tied to the accident

**What it asks:** Whether any claimant started counseling, psychiatric care, neuro-cognitive testing, or new or increased anxiety, depression or sleep medication that the notes tie to the accident, and whether that claimant had psychiatric care before.

**Question:** Has any claimant on this claim started counseling, psychiatric care, neuro-cognitive testing, or new or increased anxiety, depression or sleep medication that the notes tie to the accident? Prior history means psychiatric care before the accident. If claimants differ, pick Yes, new.

**Allowed values (copy exactly one, character for character):**
- `No`: choose when no accident-related psychological care for anyone
- `Yes, new after the accident`: choose when such care started, and that claimant had no prior psychiatric care
- `Yes, with prior psychiatric history`: choose when such care started, and that claimant had psychiatric care before

### S31: Signs of claim-driven treatment

**What it asks:** How many of nine signs of claim-driven treatment appear for any claimant, as of the latest note, each counted once: (1) no ER visit on the accident day, but a clinic visit within 1-3 days after; (2) a clinic assignment of benefits filed with the PIP application; (3) delayed treatment followed by aggressive treatment (first treatment 2+ weeks after the accident, then frequent visits, injections or many therapies); (4) attorney-referred providers (treatment begins only after the claimant hired a lawyer, or the lawyer sent them to the provider); (5) treatment far heavier than the crash suggests (injections, a surgery recommendation or months of therapy after a minor crash with a drivable car and no airbags); (6) only subjective pain complaints with no objective findings (no fracture, normal or no imaging, normal tests), yet treatment continues; (7) cookie-cutter medical records (notes say records are templated or nearly identical across visits or across claimants); (8) treatment that keeps going until the case resolves (no discharge, treatment continuing through the demand or negotiation with no medical reason noted); (9) a major injury not reported at the first doctor visit (for example a shoulder tear or disc herniation first claimed at a later visit).

**Question:** As of the latest note, how many of these nine signs apply to any claimant on this claim, counting each sign once: (1) no ER visit on the accident day, but a clinic visit within 1-3 days after; (2) a clinic assignment of benefits filed with the PIP application; (3) delayed treatment followed by aggressive treatment (first treatment 2+ weeks after the accident, then frequent visits, injections or many therapies); (4) attorney-referred providers (treatment begins only after the claimant hired a lawyer, or the lawyer sent them to the provider); (5) treatment far heavier than the crash suggests (injections, a surgery recommendation or months of therapy after a minor crash with a drivable car and no airbags); (6) only subjective pain complaints with no objective findings (no fracture, normal or no imaging, normal tests), yet treatment continues; (7) cookie-cutter medical records (notes say records are templated or nearly identical across visits or across claimants); (8) treatment that keeps going until the case resolves (no discharge, treatment continuing through the demand or negotiation with no medical reason noted); (9) a major injury not reported at the first doctor visit (for example a shoulder tear or disc herniation first claimed at a later visit)? Count a sign only when the notes describe it; do not infer it. A clinic in sign (1) includes urgent care, a chiropractor, a physical therapist or a doctor's office. Decide each sign yes or no on its own before counting.

**Allowed values (copy exactly one, word for word):**
- `None of these`: choose when none of the nine signs appear
- `One of these`: choose when exactly one of the nine signs appears
- `Two of these`: choose when exactly two of the nine signs appear
- `Three or more of these`: choose when three or more of the nine signs appear

### S32: Damages claimed beyond medical bills

**What it asks:** Within 180 days of the accident, how many of ten damage types beyond medical bills are claimed for any claimant, each counted once: (1) lost wages; (2) lost job; (3) household or essential services; (4) inability to do prior activities; (5) a spouse's companionship claim (loss of consortium); (6) out-of-pocket costs, such as transportation or medical equipment; (7) emotional distress, such as anxiety or depression; (8) funeral expenses; (9) wrongful death; (10) future damages, such as future medical care or future lost earnings. Claimed, not necessarily paid.

**Question:** Within 180 days, how many of these ten damage types are claimed for any claimant on this claim, counting each type once: (1) lost wages; (2) lost job; (3) household or essential services; (4) inability to do prior activities; (5) a spouse's companionship claim (loss of consortium); (6) out-of-pocket costs, such as transportation or medical equipment; (7) emotional distress, such as anxiety or depression; (8) funeral expenses; (9) wrongful death; (10) future damages, such as future medical care or future lost earnings? Count claimed, not paid. Transportation and medical equipment both count as the single type out-of-pocket costs.

**Allowed values (copy exactly one, word for word):**
- `None of these`: choose when none of the ten types are claimed
- `One of these`: choose when exactly one of the ten types is claimed
- `Two of these`: choose when exactly two of the ten types are claimed
- `Three or more of these`: choose when three or more of the ten types are claimed

### S33: What the company's early medical review targeted

**What it asks:** If our company ordered an IME, a records or films review, a peer review or a nurse case-manager review for any claimant within the first 180 days, what its main question was: ending routine therapy, whether the accident caused the injury, or approving a surgery or procedure.

**Question:** If the company ordered an independent medical exam, records or films review, peer review or nurse case-manager review for any claimant within the first 180 days, what was its main question? If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no such review ordered for anyone
- `Ending routine therapy`: choose when whether ongoing therapy or chiropractic is still needed
- `Whether the accident caused the injury`: choose when causation, a prior condition, or accident-relatedness
- `Approving a surgery or procedure`: choose when whether a recommended surgery or injection is needed

### S34: Type of fracture

**What it asks:** The most serious fracture documented within 90 days of the accident for any claimant, from a hairline fracture treated with a brace up to multiple surgically fixed fractures or any spine or pelvis fracture.

**Question:** What is the most serious fracture documented within 90 days among all injured claimants on this claim? Surgically fixed means plates, screws, pins or reduction under anesthesia; also use that level for a fracture into a joint, comminuted, or through a growth plate. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `None`: choose when no fracture documented for anyone
- `Non-displaced, brace only`: choose when hairline or non-displaced, treated with a cast, brace, sling or rest
- `Displaced, no surgery`: choose when displaced, but set or healing without surgery
- `Surgically fixed, joint or growth-plate`: choose when surgery or reduction under anesthesia, or into a joint, comminuted or growth plate
- `Multiple surgical, spine or pelvis`: choose when two or more surgically fixed fractures, or any spine or pelvis fracture

### S35: Claimant's claim is only derivative

**What it asks:** Whether the scored claimant's claim is only derivative (for example a spouse claiming loss of companionship, a family member claiming emotional distress, or a wrongful-death claim) rather than for their own physical injury, or both.

**Question:** Is the claimant's claim only derivative, such as a spouse suing only for loss of companionship or a family member who was not struck claiming emotional distress, rather than for their own physical injury? A claimant who was in a vehicle or was struck and claims for their own injury is `Own injury`; if every injured person claims for their own injury, answer `Own injury` even when you cannot tell which one is the claimant.

**Allowed values (copy exactly one, character for character):**
- `Own injury`: choose when claims only for their own physical injury
- `Derivative only`: choose when claims only lost companionship, emotional distress or wrongful death, no own injury
- `Both`: choose when claims their own injury plus a derivative claim
- `Cannot tell`: choose when unclear which claimant the notes mean, or what the claim is based on

### S36: Was the claimant's injury reported at first notice, or raised later

**What it asks:** Whether the first report of the accident (first notice of loss) said the scored claimant was hurt, or whether this claimant's injury was first raised later, by anyone, including in a lawyer's letter.

**Question:** Did the first report of the accident say this claimant was hurt, or was this claimant's injury first raised later, by anyone? The first notice is the first-report note plus any note within 1 day after it. If several people are injured and they would all get the same answer, give that answer.

**Allowed values (copy exactly one, character for character):**
- `Injury reported at first notice`: choose when the first report of the accident says this claimant was hurt
- `Injury raised later`: choose when first notice said no injury or left this claimant out; the injury came up later, including by a lawyer's letter
- `Cannot tell`: choose when unclear which claimant the early notes refer to

### S37: Doubts about the claim's legitimacy

**What it asks:** How many of five doubts about the claim's legitimacy the notes raise, each counted once: the police report or a witness says 'no injuries'; a claimant's account changes; a note calls it 'staged' or 'set up'; a rider got in just before the crash; surveillance contradicts a claimant's stated limits.

**Question:** How many of these signs do the notes raise about any claimant's injury claim, counting each once: the police report or a witness says 'no injuries' at the scene; a claimant's own account changes between statements; a note calls it 'staged' or 'set up'; a rider got in just before the crash; surveillance contradicts a claimant's stated limits? For the first sign, count it when the police report, EMS, a neutral witness or the claimant at the scene says no one was hurt; our insured's or the IV driver's word alone does not count. A changing account counts only for a claimant, not for our insured.

**Allowed values (copy exactly one, character for character):**
- `0`: choose when none of the five signs
- `1`: choose when exactly one sign
- `2`: choose when exactly two signs
- `3+`: choose when three or more of the five signs

### S38: How the vehicles collided

**What it asks:** What kind of collision happened, using the first impact if there were several: a pedestrian or cyclist struck, head-on, broadside (T-bone), into a fixed object, rear-end or chain, or sideswipe or parking lot.

**Question:** What kind of collision happened in this accident? If a pedestrian or cyclist was struck, pick that; if there were several impacts, use the first impact. Chain means three or more vehicles in a rear-end sequence; broadside means one vehicle's front struck another's side at an intersection or driveway.

**Allowed values (copy exactly one, character for character):**
- `Pedestrian or cyclist struck`: choose when a vehicle hit a person on foot or on a bicycle
- `Head-on`: choose when vehicle fronts collided, including wrong-way crashes
- `Broadside at intersection`: choose when front into side at an intersection or driveway, a T-bone
- `Into a fixed object`: choose when a vehicle hit a pole, tree, wall, guardrail or similar
- `Rear-end or chain`: choose when front into rear, including chains of three or more vehicles
- `Sideswipe or parking lot`: choose when sides scraped, a lane-change crash, or any parking-lot or backing crash
- `Not enough facts`: choose when notes don't say how the vehicles came together

### S39: Lawsuit filed early, before any demand

**What it asks:** If any claimant has filed a lawsuit, when the first suit was filed: early before any demand, after negotiation stalled, or near the deadline to sue. Apply the tie-break order in the question.

**Question:** If any claimant on this claim has sued, was the first suit filed within 180 days of the accident before any demand or medical package, after negotiation stalled, or within 90 days of the filing deadline? Pick Suit near the deadline first, then Early suit before any demand, otherwise Suit after negotiation stalled.

**Allowed values (copy exactly one, character for character):**
- `No suit`: choose when no lawsuit filed for any claimant
- `Early suit before any demand`: choose when filed within 180 days, before any demand or medical package
- `Suit after negotiation stalled`: choose when filed after demands or offers failed to resolve it
- `Suit near the deadline`: choose when filed within 90 days of the filing deadline
- `Not enough facts`: choose when a suit was filed, but its timing against demands and deadline is unclear

### S40: First offer amount

**What it asks:** Our company's first dollar settlement offer to each claimant, taking the largest of these first offers. A global first offer counts as one offer. Never use a later offer, an authority figure or an agreed amount.

**Question:** For each claimant on this claim, take the company's first dollar settlement offer; what is the largest of these first offers? A global first offer counts as one offer. Never use a later offer, an authority figure or an agreed amount. If several fit, pick the highest.

**Allowed values (copy exactly one, character for character):**
- `No offer yet`: choose when no dollar offer made to anyone
- `Under $5,000`: choose when largest first offer below $5,000
- `$5,000-$24,999`: choose when largest first offer $5,000 to $24,999
- `$25,000 or more`: choose when largest first offer $25,000 or more

### S41: First offer framed as nuisance or defense cost

**What it asks:** Whether our first offer to any claimant was described as nuisance value, cost of defense or 'to close the file', rather than based on what the injury is worth.

**Question:** Was the company's first offer to any claimant on this claim described as nuisance value, cost of defense or "to close the file", rather than an offer based on the value of the injury?

**Allowed values (copy exactly one, character for character):**
- `Yes`: choose when any claimant's first offer called nuisance value, cost of defense or "to close the file"
- `No`: choose when first offers were made, and none is framed that way
- `No offer yet`: choose when no offer made to any claimant

### S42: Another party mainly at fault

**What it asks:** From the police report and calls with other carriers in the first 60 days after the accident, whether another driver or party with its own insurer is mainly (over 50%) at fault, so our coverage pays only a share.

**Question:** From the police report and carrier calls in the first 60 days, is another driver or party with its own insurer mainly (over 50%) at fault, so our coverage is asked only for a share? Shared evenly means about 50/50.

**Allowed values (copy exactly one, character for character):**
- `Our insured sole or main at fault`: choose when IV driver is over 50% or wholly at fault
- `Shared evenly`: choose when fault stated as 50/50 or equal; any split above 50% for one side goes to that side's option
- `Another party main, we contribute`: choose when another party over 50% at fault; IV shares some
- `Another party wholly at fault`: choose when another party 100% at fault; IV has no share
- `Not enough facts`: choose when no police report or fault discussion in the first 60 days

### S43: Early shoulder, knee, hip or wrist complaint

**What it asks:** Within 30 days of the accident, whether any claimant reported pain or injury to a shoulder, knee, hip or wrist, rather than only the neck or back.

**Question:** Within 30 days of the accident, did any claimant on this claim report pain or injury to a shoulder, knee, hip or wrist, rather than only the neck or back? A collarbone or rotator cuff counts as the shoulder. Use `Neck or back only` when body-part complaints are recorded in the first 30 days but none is a shoulder, knee, hip or wrist (for example head, chest, arm, hand or leg). Use `Not enough facts` only when no body-part complaint is recorded in the first 30 days.

**Allowed values (copy exactly one, character for character):**
- `Joint complaint reported`: choose when any claimant reports shoulder, knee, hip or wrist pain or injury by day 30
- `Neck or back only`: choose when complaints in 30 days are limited to neck, back or other non-joint areas
- `Not enough facts`: choose when no body-part complaints recorded in the first 30 days

# Output format

Return exactly one JSON object with 171 keys, in this order: A01-A59, then L01-L69, then S01-S43. Each key holds an object with two fields:
- `evidence`: a string, at most 25 words.
- `value`: one allowed value for that feature, copied exactly. For A21 it is an integer.

```json
{
  "A01": {"evidence": "", "value": "Yes"},
  "A02": {"evidence": "", "value": "Yes"},
  "A03": {"evidence": "", "value": "Yes"},
  "A21": {"evidence": "", "value": -1},
  "...": "one entry for every feature through S43"
}
```


# Claim data for this snapshot

<claim_facts>
Today's date (snapshot date; use this as today): {SNAPSHOT_DATE}
Accident date: {LOSS_DATE}
Accident state: {LOSS_STATE}
Policy state: {POL_STATE}
BI type of the coverage being scored: {BI_TYPE}
Claimant being scored: {CLAIMANT}
</claim_facts>

<claim_notes>
{CLAIM_NOTES}
</claim_notes>

Now answer all 171 features for this claim. Read all of the notes before you answer. For each feature, check the question, the "choose when" definitions and the tie-break, then copy one allowed value exactly. Before you finish, check that every `value` matches an entry in its feature's allowed list character for character, and that related answers agree. Return only the JSON object.
'''

print(f'{len(PROMPT):,} characters')

192,463 characters


In [4]:
DATA_MARKER = '# Claim data for this snapshot'
PLACEHOLDERS = ['SNAPSHOT_DATE', 'LOSS_DATE', 'LOSS_STATE', 'POL_STATE', 'BI_TYPE', 'CLAIMANT']

# fixed instructions -> system prompt (cached); claim data part -> filled per snapshot
_i = PROMPT.index(DATA_MARKER)
SYSTEM_PROMPT, DATA_TEMPLATE = PROMPT[:_i].rstrip() + '\n', PROMPT[_i:]

# allowed values per feature, parsed from the prompt itself: {key: [values]} or 'INTEGER' (A21)
SPEC = {}
for m in re.finditer(r'^### ([ALS]\d\d): [^\n]*\n(.*?)(?=^### |^## |^# |\Z)', PROMPT, flags=re.M | re.S):
    key, body = m.group(1), m.group(2)
    SPEC[key] = ('INTEGER' if 'only feature whose value is a JSON integer' in body
                 else re.findall(r'^- `([^`]*)`', body.split('**Allowed values', 1)[1], flags=re.M))

SCHEMA = {'type': 'object', 'required': list(SPEC), 'additionalProperties': False, 'properties': {
    k: {'type': 'object', 'required': ['evidence', 'value'], 'additionalProperties': False,
        'properties': {'evidence': {'type': 'string'},
                       'value': {'type': 'integer'} if v == 'INTEGER' else {'type': 'string', 'enum': v}}}
    for k, v in SPEC.items()}}

os.makedirs(os.path.join(OUTPUT_DIR, '_failed'), exist_ok=True)
os.makedirs(os.path.dirname(SYSTEM_PROMPT_FILE), exist_ok=True)
with open(SYSTEM_PROMPT_FILE, 'w') as f:
    f.write(SYSTEM_PROMPT)
WORKDIR = tempfile.mkdtemp(prefix='agent_pilot_')   # empty working folder: the agent sees no project files

sample = pd.read_parquet(SAMPLE_PATH)
print(f'{len(SPEC)} LLM features | system prompt {len(SYSTEM_PROMPT):,} chars | schema {len(json.dumps(SCHEMA)):,} chars')
print(f'{len(sample)} snapshots:', sample.MODEL.value_counts().to_dict())

171 LLM features | system prompt 191,740 chars | schema 45,785 chars
100 snapshots: {'Attorney': 34, 'Litigation': 33, 'Severity': 33}


In [5]:
def fill_data(row):
    """Claim-data part of the prompt for one snapshot. Notes go in last, so braces inside notes are safe."""
    text = DATA_TEMPLATE
    for ph in PLACEHOLDERS:
        v = row[ph]
        text = text.replace('{' + ph + '}', 'not provided' if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    return text.replace('{CLAIM_NOTES}', row['NOTE_TEXT'])


def validate(answer):
    """Exact, case-sensitive check of every value. Returns a list of problems (empty = valid)."""
    if not isinstance(answer, dict):
        return ['answer is not a JSON object']
    problems = []
    if [k for k in SPEC if k not in answer]:
        problems.append(f'missing keys: {[k for k in SPEC if k not in answer]}')
    if [k for k in answer if k not in SPEC]:
        problems.append(f'unexpected keys: {[k for k in answer if k not in SPEC]}')
    for key, vals in SPEC.items():
        item = answer.get(key)
        if item is None:
            continue
        if not isinstance(item, dict) or 'value' not in item:
            problems.append(f'{key}: not an object with a value')
            continue
        v = item['value']
        if vals == 'INTEGER':
            if not (isinstance(v, int) and not isinstance(v, bool) and (v in (-1, -99) or v >= 0)):
                problems.append(f'{key}: {v!r} is not -1, -99 or a whole number >= 0')
        elif v not in vals:
            problems.append(f'{key}: {v!r} is not an allowed value')
    return problems


def output_path(row):
    return os.path.join(OUTPUT_DIR, f"{row['MODEL']}_{int(row['CLM_CVG_PK'])}_snap{int(row['SNAPSHOT_NUM'])}.json")


def make_options():
    return ClaudeAgentOptions(
        model=MODEL,
        cli_path=CLAUDE_CODE,
        system_prompt={'type': 'file', 'path': SYSTEM_PROMPT_FILE},
        tools=[],                    # no Claude Code tools: the model only reads the notes and answers
        setting_sources=[],          # ignore CLAUDE.md and user/project settings
        cwd=WORKDIR,
        max_turns=MAX_TURNS,
        thinking={'type': 'adaptive'},
        effort=EFFORT,
        output_format={'type': 'json_schema', 'schema': SCHEMA},
        verbatim_prompts=True,       # deliver the notes exactly as written (no @file or /command handling)
        max_budget_usd=MAX_BUDGET_USD_PER_CALL,
    )


async def call_agent(user_prompt):
    """One Agent SDK call. Returns the ResultMessage (or None)."""
    result = None
    async for msg in query(prompt=user_prompt, options=make_options()):
        if isinstance(msg, ResultMessage):
            result = msg
    return result


def usage_record(res):
    u, mu = res.usage or {}, res.model_usage or {}
    reported = [m.get('thinkingTokens') for m in mu.values() if m.get('thinkingTokens') is not None]
    if not reported and (u.get('output_tokens_details') or {}).get('thinking_tokens') is not None:
        reported = [u['output_tokens_details']['thinking_tokens']]
    cc = u.get('cache_creation') or {}
    return {'input_tokens': u.get('input_tokens', 0),
            'cache_creation_input_tokens': u.get('cache_creation_input_tokens', 0),
            'cache_write_5m_tokens': cc.get('ephemeral_5m_input_tokens', 0),
            'cache_write_1h_tokens': cc.get('ephemeral_1h_input_tokens', 0),
            'cache_read_input_tokens': u.get('cache_read_input_tokens', 0),
            'output_tokens': u.get('output_tokens', 0),            # includes thinking tokens
            'thinking_tokens': sum(reported) if reported else None,
            'total_cost_usd': res.total_cost_usd,                  # SDK's client-side estimate
            'duration_ms': res.duration_ms, 'num_turns': res.num_turns,
            'subtype': res.subtype, 'is_error': res.is_error, 'model_usage': mu}


async def extract_one(row):
    """Extract all features for one snapshot and save output/<MODEL>_<CLM_CVG_PK>_snap<N>.json."""
    base_prompt = fill_data(row)
    user_prompt, attempts, calls, answer, problems = base_prompt, [], [], None, ['not run']
    for _ in range(MAX_ATTEMPTS):
        res = await call_agent(user_prompt)
        if res is None:
            problems = ['no result message returned']
        else:
            calls.append(usage_record(res))
            answer = res.structured_output
            problems = (validate(answer) if not res.is_error and answer is not None
                        else [f'run failed: subtype={res.subtype} errors={res.errors}'])
        attempts.append(problems)
        if not problems:
            break
        user_prompt = (base_prompt + '\n\nYour previous answer had these problems. Fix them and return the full JSON '
                       'again, copying every value exactly from its allowed list:\n- ' + '\n- '.join(problems))

    snap = {c: row[c] for c in ['MODEL', 'CLM_CVG_PK', 'CLM_NUM', 'SNAPSHOT_NUM', 'SNAPSHOT_DAY', 'SNAPSHOT_DATE',
                                'N_NOTES', 'NOTE_CHARS']}
    record = {'snapshot': snap,
              'run': {'model': MODEL, 'effort': EFFORT, 'attempts': len(attempts), 'problems_per_attempt': attempts,
                      'valid': not problems, 'finished_at': datetime.now(timezone.utc).isoformat(timespec='seconds')},
              'usage_per_call': calls}
    if not problems:
        record['features'] = {**{k: answer[k]['value'] for k in SPEC}, 'A60': row['A60'], 'L70': row['L70']}
        record['evidence'] = {k: answer[k]['evidence'] for k in SPEC}
        path = output_path(row)
    else:
        record['last_answer'] = answer
        path = os.path.join(OUTPUT_DIR, '_failed', os.path.basename(output_path(row)))
    with open(path, 'w') as f:
        json.dump(record, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    return record, path


def call_summary(record):
    c = record['usage_per_call']
    return {'valid': record['run']['valid'], 'attempts': record['run']['attempts'],
            'input': sum(x['input_tokens'] + x['cache_creation_input_tokens'] + x['cache_read_input_tokens'] for x in c),
            'output': sum(x['output_tokens'] for x in c),
            'thinking': sum(x['thinking_tokens'] or 0 for x in c),
            'cost_usd': round(sum(x['total_cost_usd'] or 0 for x in c), 4)}

## 4. Token estimate before running
Input tokens per snapshot = system prompt + claim data, estimated from characters (about 2.2 characters per token,
measured on a real call). Cost is split by token type:
- **input** — the ~80k-token system prompt read from cache ($0.20 / M) plus the snapshot's claim data written to the
  1-hour cache ($8 / M); the very first call also writes the system prompt to the cache once.
- **thinking** — the model's reasoning, billed as output ($20 / M).
- **answer** — the visible JSON (171 values + evidence), billed as output ($20 / M).

Thinking and answer sizes are guesses until section 7 measures them.

In [6]:
CHARS_PER_TOKEN = 2.2
ASSUMED_THINKING_TOKENS = 3_000    # first real call: 2.5k thinking
ASSUMED_ANSWER_TOKENS = 9_000      # first real call: 7k visible JSON (171 values + evidence); more notes -> more evidence

est = sample[['MODEL', 'CLM_CVG_PK', 'SNAPSHOT_NUM', 'N_NOTES', 'NOTE_CHARS']].copy()
system_tokens = round(len(SYSTEM_PROMPT) / CHARS_PER_TOKEN)
est['data_tokens'] = [round(len(fill_data(r)) / CHARS_PER_TOKEN) for _, r in sample.iterrows()]
est['input_tokens'] = system_tokens + est.data_tokens
# system prompt read from cache; the claim data is new on every call (written to the 1-hour cache)
est['input_cost'] = (system_tokens * PRICES['cache_read'] + est.data_tokens * PRICES['cache_write_1h']) / 1e6
est['thinking_cost'] = ASSUMED_THINKING_TOKENS * PRICES['output'] / 1e6
est['answer_cost'] = ASSUMED_ANSWER_TOKENS * PRICES['output'] / 1e6
est['total_cost'] = est.input_cost + est.thinking_cost + est.answer_cost
first_write = system_tokens * PRICES['cache_write_1h'] / 1e6
print(f'system prompt ~{system_tokens:,} tokens; the first call writes it to the cache: ~${first_write:.2f} once')
display(est[['N_NOTES', 'input_tokens', 'data_tokens', 'input_cost', 'thinking_cost', 'answer_cost', 'total_cost']]
        .describe().loc[['mean', '50%', 'max']].round(3))
tot = est[['input_cost', 'thinking_cost', 'answer_cost']].sum()
print(f'estimated total for {len(est)} snapshots: ~${tot.sum() + first_write:.0f} = input ${tot.input_cost + first_write:.0f} '
      f'+ thinking ${tot.thinking_cost:.0f} + answer ${tot.answer_cost:.0f}')


system prompt ~87,155 tokens; the first call writes it to the cache: ~$0.70 once


,N_NOTES,input_tokens,data_tokens,input_cost,thinking_cost,answer_cost,total_cost
mean,23.3,96666.4,9511.4,0.094,0.06,0.18,0.334
50%,14.5,94743.0,7588.0,0.078,0.06,0.18,0.318
max,91.0,119259.0,32104.0,0.274,0.06,0.18,0.514


estimated total for 100 snapshots: ~$34 = input $10 + thinking $6 + answer $18


## 5. Test on one snapshot
Runs the first snapshot (or shows its saved result if it was already run). Check `valid`, the tokens and a few
answers before running everything.

In [7]:
row = sample.iloc[0]
if os.path.exists(output_path(row)):
    record = json.load(open(output_path(row)))
    print('already run ->', output_path(row))
else:
    t0 = time.time()
    record, path = await extract_one(row)
    print(f'{time.time() - t0:.0f}s ->', path)
print(call_summary(record))
if record['run']['valid']:
    display(pd.DataFrame({'value': record['features'], 'evidence': record.get('evidence', {})}).head(25))
else:
    print(record['run']['problems_per_attempt'])

94s -> output/Attorney_52939746_snap4.json
{'valid': True, 'attempts': 1, 'input': 90419, 'output': 12374, 'thinking': 4372, 'cost_usd': 0.2691}


## 6. Run all snapshots — 10 in parallel
1. **Warm the cache:** the first pending snapshot runs alone, so the ~80k-token system prompt is written to the cache once.
2. **Then 10 at a time:** the rest run in parallel (`PARALLEL = 10`) and all read the prompt from the cache.

Every line shows the counter `[done/total]`, plus ok/failed so far. Lines print as each snapshot finishes, so the
snapshot order is mixed. Snapshots that already have an output file are skipped, so it is safe to stop and re-run.
If calls fail with rate-limit errors, lower `PARALLEL` and re-run: only the missing snapshots are redone.

In [8]:
import asyncio

PARALLEL = 10

pending = [row for _, row in sample.iterrows() if not os.path.exists(output_path(row))]
total = len(sample)
progress = {'done': total - len(pending), 'ok': 0, 'failed': 0, 'cost': 0.0}
print(f'{progress["done"]} of {total} already saved, {len(pending)} to run')


async def run_snapshot(row, sem=None):
    t0 = time.time()
    try:
        if sem is None:
            record, _ = await extract_one(row)
        else:
            async with sem:
                t0 = time.time()
                record, _ = await extract_one(row)
        s = call_summary(record)
        progress['ok' if s['valid'] else 'failed'] += 1
        progress['cost'] += s['cost_usd']
        status = (f"{'ok' if s['valid'] else 'FAILED'} in {time.time() - t0:.0f}s | in {s['input']:,} "
                  f"out {s['output']:,} thinking {s['thinking']:,} | ${s['cost_usd']:.3f}")
    except Exception as e:                      # keep going; the snapshot is redone on the next run
        progress['failed'] += 1
        status = f'ERROR {type(e).__name__}: {str(e)[:200]}'
    progress['done'] += 1
    print(f"[{progress['done']}/{total}] ok {progress['ok']} failed {progress['failed']} | "
          f"{row.MODEL} {row.CLM_CVG_PK} snap{row.SNAPSHOT_NUM}: {status} | running ${progress['cost']:.2f}", flush=True)


t_start = time.time()
if pending:
    print('warming the cache with one snapshot ...', flush=True)
    await run_snapshot(pending[0])
if len(pending) > 1:
    print(f'cache warm - running the other {len(pending) - 1} snapshots, {PARALLEL} at a time ...', flush=True)
    sem = asyncio.Semaphore(PARALLEL)
    await asyncio.gather(*(run_snapshot(row, sem) for row in pending[1:]))

saved = len(glob.glob(os.path.join(OUTPUT_DIR, '*.json')))
print(f"\ndone in {(time.time() - t_start) / 60:.1f} min: {progress['ok']} ok, {progress['failed']} failed this run, "
      f"~${progress['cost']:.2f} | {saved} of {total} snapshots now in {OUTPUT_DIR}/")


1 of 100 already saved, 99 to run
warming the cache with one snapshot ...


[2/100] ok 1 failed 0 | Attorney 55910442 snap1: ok in 65s | in 85,202 out 9,285 thinking 2,619 | $0.206 | running $0.21


cache warm - running the other 98 snapshots, 10 at a time ...


[3/100] ok 2 failed 0 | Attorney 98945653 snap1: ok in 67s | in 85,253 out 9,437 thinking 2,390 | $0.218 | running $0.42


[4/100] ok 3 failed 0 | Attorney 75894952 snap3: ok in 73s | in 86,451 out 10,097 thinking 2,764 | $0.241 | running $0.66


[5/100] ok 4 failed 0 | Attorney 87435731 snap1: ok in 79s | in 86,054 out 10,754 thinking 3,639 | $0.251 | running $0.91


... (92 outputs left out)

[98/100] ok 97 failed 0 | Severity 138442517 snap10: ok in 132s | in 93,104 out 16,564 thinking 7,503 | $0.353 | running $32.93


[99/100] ok 98 failed 0 | Severity 97223517 snap9: ok in 194s | in 100,291 out 23,309 thinking 12,653 | $0.491 | running $33.42


[100/100] ok 99 failed 0 | Severity 150627771 snap9: ok in 231s | in 109,540 out 26,512 thinking 16,754 | $0.557 | running $33.98



done in 24.1 min: 99 ok, 0 failed this run, ~$33.98 | 100 of 100 snapshots now in output/


## 7. Tokens and cost per call, per snapshot
Real usage of every call, from the saved JSONs (including `output/_failed`), split into **input**, **thinking** and
**answer** tokens, each with its own cost at Opus 5.5 list prices ($4 input, $8 1-hour cache write, $0.20 cache
read, $20 output per million). Thinking tokens are reported by Claude Code; answer tokens = output - thinking.
`sdk_cost_usd` is the Agent SDK's own estimate, shown as a cross-check.

In [9]:
rows = []
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.json')) + glob.glob(os.path.join(OUTPUT_DIR, '_failed', '*.json'))):
    r = json.load(open(path))
    for n, c in enumerate(r['usage_per_call'], 1):
        rows.append({**{k: r['snapshot'][k] for k in ['MODEL', 'CLM_CVG_PK', 'SNAPSHOT_NUM', 'N_NOTES']},
                     'call': n, 'valid': r['run']['valid'],
                     'input_uncached': c['input_tokens'], 'cache_write_5m': c.get('cache_write_5m_tokens', 0),
                     'cache_write_1h': c.get('cache_write_1h_tokens', c['cache_creation_input_tokens']),
                     'cache_read': c['cache_read_input_tokens'], 'output_tokens': c['output_tokens'],
                     'thinking_tokens': c['thinking_tokens'] or 0, 'sdk_cost_usd': c['total_cost_usd'],
                     'seconds': c['duration_ms'] / 1000})
usage = pd.DataFrame(rows)
if usage.empty:
    print('No outputs yet - run section 5 or 6 first.')
else:
    # tokens, split by type (thinking is part of output_tokens; answer = the visible JSON)
    usage['input_tokens'] = usage.input_uncached + usage.cache_write_5m + usage.cache_write_1h + usage.cache_read
    usage['answer_tokens'] = usage.output_tokens - usage.thinking_tokens
    # cost of each token type at Opus 5.5 list prices
    usage['input_cost'] = (usage.input_uncached * PRICES['input'] + usage.cache_write_5m * PRICES['cache_write_5m']
                           + usage.cache_write_1h * PRICES['cache_write_1h'] + usage.cache_read * PRICES['cache_read']) / 1e6
    usage['thinking_cost'] = usage.thinking_tokens * PRICES['output'] / 1e6
    usage['answer_cost'] = usage.answer_tokens * PRICES['output'] / 1e6
    usage['total_cost'] = usage.input_cost + usage.thinking_cost + usage.answer_cost

    print('Tokens and cost per call')
    display(usage[['MODEL', 'CLM_CVG_PK', 'SNAPSHOT_NUM', 'N_NOTES', 'call', 'valid',
                   'input_tokens', 'cache_read', 'cache_write_1h', 'input_uncached', 'thinking_tokens', 'answer_tokens',
                   'input_cost', 'thinking_cost', 'answer_cost', 'total_cost', 'sdk_cost_usd', 'seconds']].round(4))

    tok = ['input_tokens', 'thinking_tokens', 'answer_tokens']
    cost = ['input_cost', 'thinking_cost', 'answer_cost', 'total_cost']
    by = pd.concat([usage.groupby('MODEL')[tok + cost].mean(), usage[tok + cost].mean().to_frame('ALL (mean)').T,
                    usage[tok + cost].sum().to_frame('ALL (total)').T])
    print('Mean per call by model, plus overall mean and total')
    display(by.round(3))
    share = usage[cost[:-1]].sum() / usage.total_cost.sum() * 100
    print('Share of total cost: ' + ', '.join(f"{k.replace('_cost', '')} {v:.0f}%" for k, v in share.items()))
    per_snapshot = usage.groupby(['MODEL', 'CLM_CVG_PK', 'SNAPSHOT_NUM']).total_cost.sum()
    print(f'{len(per_snapshot)} snapshots, {len(usage)} calls ({len(usage) / len(per_snapshot):.2f} per snapshot) | '
          f'cost per snapshot: mean ${per_snapshot.mean():.3f}, max ${per_snapshot.max():.3f}, total ${per_snapshot.sum():.2f}')
    print(f"SDK's own cost estimate for the same calls: ${usage.sdk_cost_usd.sum():.2f}")


Tokens and cost per call


,MODEL,CLM_CVG_PK,SNAPSHOT_NUM,N_NOTES,call,valid,input_tokens,cache_read,cache_write_1h,input_uncached,thinking_tokens,answer_tokens,input_cost,thinking_cost,answer_cost,total_cost,sdk_cost_usd,seconds
0,Attorney,101150597,6,21,1,True,93161,92822,337,2,5777,8147,0.0213,0.1155,0.1629,0.2997,0.3008,108.373
1,Attorney,103939420,3,7,1,True,87830,87491,337,2,3483,7971,0.0202,0.0697,0.1594,0.2493,0.2498,83.186
2,Attorney,106715563,3,3,1,True,85830,85491,337,2,3853,8111,0.0198,0.0771,0.1622,0.2591,0.2594,88.115
3,Attorney,107854777,2,4,1,True,90091,89752,337,2,8197,8246,0.0207,0.1639,0.1649,0.3495,0.3504,130.003
4,Attorney,116595675,1,1,1,True,85763,85424,337,2,2383,7182,0.0198,0.0477,0.1436,0.2111,0.2114,67.115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Severity,90117564,11,91,1,True,111526,111187,337,2,16986,10131,0.0249,0.3397,0.2026,0.5673,0.5701,232.616
96,Severity,92161913,4,20,1,True,91960,91621,337,2,7082,8374,0.0210,0.1416,0.1675,0.3301,0.3311,121.909
97,Severity,92269818,6,39,1,True,102843,102504,337,2,14971,9324,0.0232,0.2994,0.1865,0.5091,0.5112,203.714
98,Severity,97223517,9,65,1,True,100291,99952,337,2,12653,10656,0.0227,0.2531,0.2131,0.4889,0.4906,192.801


Mean per call by model, plus overall mean and total


,input_tokens,thinking_tokens,answer_tokens,input_cost,thinking_cost,answer_cost,total_cost
Attorney,92224.529,4798.529,8185.471,0.029,0.096,0.164,0.288
Litigation,93066.848,7793.061,8532.061,0.022,0.156,0.171,0.348
Severity,96370.121,9431.485,8885.576,0.023,0.189,0.178,0.390
ALL (mean),93870.540,7315.600,8530.880,0.025,0.146,0.171,0.341
ALL (total),9387054.000,731560.000,853088.000,2.456,14.631,17.062,34.149


Share of total cost: input 7%, thinking 43%, answer 50%
100 snapshots, 100 calls (1.00 per snapshot) | cost per snapshot: mean $0.341, max $0.741, total $34.15
SDK's own cost estimate for the same calls: $34.25


## 8. Does a second run give the same features? (compare with a previous run)
Compares this run's `output/` with an earlier run's folder (`PREVIOUS_DIR`) for the same snapshots: for every
feature, the share of snapshots where both runs chose exactly the same value. Low-agreement features are the ones
whose definitions are ambiguous and worth tightening. (Evidence quotes are not compared, only the values.)

In [10]:
PREVIOUS_DIR = 'output3'          # folder of the earlier run to compare against


def load_features(folder):
    out = {}
    for path in glob.glob(os.path.join(folder, '*.json')):
        r = json.load(open(path))
        if 'features' in r:
            out[os.path.basename(path)] = {**r['features'], '_MODEL': r['snapshot']['MODEL']}
    return out


new, old = load_features(OUTPUT_DIR), load_features(PREVIOUS_DIR)
common = sorted(set(new) & set(old))
print(f'{len(common)} snapshots in both runs (this run {len(new)}, previous run {len(old)})')
keys = list(SPEC)                  # the 171 LLM features (A60 and L70 come from data, so they always match)
same = pd.DataFrame([{k: new[f][k] == old[f][k] for k in keys} for f in common], index=common)
models = pd.Series([new[f]['_MODEL'] for f in common], index=common)

print(f'overall: {same.values.mean() * 100:.1f}% of all feature values identical; '
      f'{(same.all(axis=1)).sum()} of {len(common)} snapshots identical on every feature')
per_snapshot = same.mean(axis=1) * 100
print(f'per snapshot: median {per_snapshot.median():.1f}% identical, lowest {per_snapshot.min():.1f}%')
display((same.groupby(models).mean().mean(axis=1) * 100).round(1).to_frame('% identical, by snapshot model'))

agree = (same.mean() * 100).round(1).sort_values()
print(f'\nfeatures 100% identical: {(agree == 100).sum()} of {len(agree)}; below 90%: {(agree < 90).sum()}')
rows = []
for k in agree[agree < 100].index:
    changes = pd.Series([f'{old[f][k]}  ->  {new[f][k]}' for f in common if old[f][k] != new[f][k]]).value_counts()
    rows.append({'feature': k, '% identical': agree[k], 'n changed': int(changes.sum()),
                 'most common change (previous -> this run)': f'{changes.index[0]}  (x{changes.iloc[0]})'})
print('features that changed between the runs, least stable first:')
display(pd.DataFrame(rows).set_index('feature'))

100 snapshots in both runs (this run 100, previous run 100)
overall: 97.8% of all feature values identical; 8 of 100 snapshots identical on every feature
per snapshot: median 98.2% identical, lowest 86.5%


,"% identical, by snapshot model"
Attorney,98.9
Litigation,97.5
Severity,97.0



features 100% identical: 48 of 171; below 90%: 0
features that changed between the runs, least stable first:


,% identical,n changed,most common change (previous -> this run)
feature,,,
S43,90.0,10,Joint complaint reported -> Not enough facts...
A48,91.0,9,Not enough facts -> No (x4)
A53,92.0,8,2-3 -> 0-1 (x4)
A08,92.0,8,Reactive -> Proactive (x5)
A54,93.0,7,No -> Yes (x4)
...,...,...,...
A01,99.0,1,No -> Yes (x1)
S34,99.0,1,"Non-displaced, brace only -> None (x1)"
S23,99.0,1,"Flagged, injections or surgery followed -> N..."
